In [1]:
"""
DATAMART AUTOCONSTRUCCION (encuestas de censadores) -> SQL Server
Sigue la plantilla del datamart Northwind: conexion con pyodbc, crear esquema (borra y crea),
cargar dimensiones, cargar tabla de hechos.

Origen  : data_origen_1.xlsx  (+ oficinas.csv y peru_distrital_simple.geojson, opcionales, solo para nombres)
Destino : SQL Server, base DatamartAutoconstruccion (el script la crea si no existe)
Grano   : 1 fila de fact_consumo = 1 encuesta de censador (codencuestacensadores)

Uso:
    python codigo_completo_datamart_autoconstruccion.py              # carga completa
    python codigo_completo_datamart_autoconstruccion.py --dry-run    # solo prepara y valida en memoria, no toca SQL Server

Requiere: pip install pyodbc pandas openpyxl     (y ODBC Driver 17 for SQL Server)
"""
import os
import sys
import json
import datetime as dt

import numpy as np
import pandas as pd
import pyodbc

# ----------------------------------------------------------------------------- CONFIGURACION
RUTA_DATOS = "data_origen_1.xlsx"
RUTA_OFICINAS = "oficinas.csv"                     # opcional: nombres de oficina
RUTA_GEOJSON = "peru_distrital_simple.geojson"     # opcional: nombres de departamento/provincia/distrito

SERVER = os.getenv("SQL_SERVER", "localhost")
DATABASE = "DatamartAutoconstruccion"
UID = os.getenv("SQL_UID", "sa")
PWD = os.getenv("SQL_PWD", "1234567890")           # recomendado: definir SQL_PWD como variable de entorno

FECHA_INI = dt.date(2023, 1, 1)                    # rango de la dimension Tiempo
FECHA_FIN = dt.date(2031, 12, 31)

# codigomarcalocal que se considera MARCA PROPIA (1 = "Cemex Cemento", antes "Pacasmayo Cemento").
# Si Pacasmayo Concreto (15), Rapimix (28), Bloques/Adoquines Pacasmayo (23, 25) tambien son propias, agregarlas aqui.
MARCAS_PROPIAS = {1}

SIN_DESC = "SIN DESCRIPCION"
LOTE = 20000                                       # filas por executemany

# columnas del origen que NO se cargan
DESCARTAR = ["Unnamed: 0", "desobservacionom_encuesta", "fecha_creacion_visita"]

MESES = ["Enero", "Febrero", "Marzo", "Abril", "Mayo", "Junio", "Julio", "Agosto",
         "Setiembre", "Octubre", "Noviembre", "Diciembre"]
DIAS = ["Lunes", "Martes", "Miercoles", "Jueves", "Viernes", "Sabado", "Domingo"]


# ----------------------------------------------------------------------------- CONEXIONES
def _cadena(database):
    return (
        "DRIVER={ODBC Driver 17 for SQL Server};"
        f"SERVER={SERVER};DATABASE={database};UID={UID};PWD={PWD}"
    )


def crear_base_si_no_existe():
    try:
        conn = pyodbc.connect(_cadena("master"), autocommit=True)   # CREATE DATABASE no corre dentro de una transaccion
        conn.cursor().execute(f"IF DB_ID(N'{DATABASE}') IS NULL CREATE DATABASE [{DATABASE}]")
        conn.close()
        return True
    except pyodbc.Error as e:
        print(f"Error al crear/verificar la base: {e}")
        return False


def get_connection_destino():
    try:
        conn = pyodbc.connect(_cadena(DATABASE))
        print("✓ Conexión a destino exitosa")
        return conn
    except pyodbc.Error as e:
        print(f"Error de conexión al destino: {e}")
        return None


# ----------------------------------------------------------------------------- ESQUEMA
# Orden de borrado: primero el hecho, luego las dimensiones
TABLAS = ["fact_consumo", "Competencia", "Proveedor", "Ejecutivo", "Motivo", "UnidadMedida",
          "Marca", "TipoEstructura", "Obra", "Ubigeo", "Oficina", "Tiempo"]

DDL = [
    """CREATE TABLE Tiempo (
        id INT NOT NULL PRIMARY KEY,                 -- yyyymmdd (igual a iddia del origen)
        fecha DATE NOT NULL UNIQUE,
        anio SMALLINT NOT NULL,
        trimestre TINYINT NOT NULL,
        numMes TINYINT NOT NULL,
        mes NVARCHAR(12) NOT NULL,
        anioMes INT NOT NULL,                        -- yyyymm
        dia TINYINT NOT NULL,
        semanaISO TINYINT NOT NULL,
        diaSemana TINYINT NOT NULL,                  -- 1 = lunes ... 7 = domingo
        nombreDia NVARCHAR(12) NOT NULL,
        esFinDeSemana BIT NOT NULL
    )""",
    """CREATE TABLE Oficina (
        id INT IDENTITY(1,1) PRIMARY KEY,
        codigo INT NOT NULL UNIQUE,                  -- idoficina
        nombre NVARCHAR(100) NOT NULL,
        idSociedad INT NULL
    )""",
    """CREATE TABLE Ubigeo (
        id INT IDENTITY(1,1) PRIMARY KEY,
        codigo CHAR(6) NOT NULL UNIQUE,              -- ubigeo INEI: dep(2) + prov(2) + dist(2)
        codDepartamento INT NOT NULL,
        codProvincia INT NOT NULL,
        codDistrito INT NOT NULL,
        departamento NVARCHAR(100) NOT NULL,
        provincia NVARCHAR(100) NOT NULL,
        distrito NVARCHAR(100) NOT NULL
    )""",
    """CREATE TABLE Obra (
        id INT IDENTITY(1,1) PRIMARY KEY,
        codigo BIGINT NOT NULL UNIQUE,               -- codigoobra (supera el rango de INT)
        idMaestroObra INT NULL,                      -- una obra llega a tener varios: se toma el de la encuesta mas reciente
        tipoEjecucion NVARCHAR(50) NOT NULL
    )""",
    """CREATE TABLE TipoEstructura (
        id INT IDENTITY(1,1) PRIMARY KEY,
        codigo INT NOT NULL UNIQUE,                  -- id_tipo_estructura
        nombre NVARCHAR(100) NOT NULL
    )""",
    # Marca: una fila por codigotipomarcalocal (determina marca, linea, sublinea y producto)
    """CREATE TABLE Marca (
        id INT IDENTITY(1,1) PRIMARY KEY,
        codigo INT NOT NULL UNIQUE,                  -- codigotipomarcalocal
        tipoMarca NVARCHAR(100) NOT NULL,            -- destipomarca (NO usar para analizar marcas)
        codMarca INT NULL,
        marca NVARCHAR(100) NOT NULL,                -- desmarca tal como llega
        marcaNormalizada NVARCHAR(100) NOT NULL,     -- UPPER(desmarca): une 'Holcim' y 'HOLCIM'
        esMarcaPropia BIT NOT NULL,
        codLinea INT NULL,
        linea NVARCHAR(100) NOT NULL,
        codSublinea INT NULL,
        sublinea NVARCHAR(100) NOT NULL,
        codProducto INT NULL,
        producto NVARCHAR(100) NOT NULL
    )""",
    """CREATE TABLE UnidadMedida (
        id INT IDENTITY(1,1) PRIMARY KEY,
        codigo INT NOT NULL UNIQUE,
        abreviatura NVARCHAR(20) NOT NULL
    )""",
    """CREATE TABLE Motivo (
        id INT IDENTITY(1,1) PRIMARY KEY,
        codigo INT NOT NULL UNIQUE,
        nombre NVARCHAR(100) NOT NULL                -- el codigo 12 llega sin descripcion
    )""",
    """CREATE TABLE Ejecutivo (
        id INT IDENTITY(1,1) PRIMARY KEY,
        codigo INT NOT NULL UNIQUE,                  -- codcreadorom (1:1 con ejecutivosapvisita)
        ejecutivoSAP INT NULL,
        nombre NVARCHAR(150) NOT NULL
    )""",
    """CREATE TABLE Proveedor (
        id INT IDENTITY(1,1) PRIMARY KEY,
        codigo BIGINT NOT NULL UNIQUE,               -- codisicom_isicom_proveedor (12 digitos)
        nombre NVARCHAR(200) NOT NULL                -- no viene en el dataset: completar desde comercial_lcm_cliente_isicom
    )""",
    """CREATE TABLE Competencia (
        id INT IDENTITY(1,1) PRIMARY KEY,
        codTipoCompetencia INT NOT NULL,             -- 0 = no aplica
        tipoCompetencia NVARCHAR(100) NOT NULL,
        codMarcaCemento INT NOT NULL,                -- 0 = no aplica
        marcaCemento NVARCHAR(100) NOT NULL,
        CONSTRAINT UQ_Competencia UNIQUE (codTipoCompetencia, codMarcaCemento)
    )""",
    """CREATE TABLE fact_consumo (
        id INT NOT NULL PRIMARY KEY,                 -- codencuestacensadores
        visitaID INT NOT NULL,                       -- codvisita (dimension degenerada)
        tiempoID INT NOT NULL,
        oficinaID INT NOT NULL,
        ubigeoID INT NOT NULL,
        obraID INT NOT NULL,
        tipoEstructuraID INT NOT NULL,
        marcaID INT NOT NULL,
        unidadMedidaID INT NOT NULL,
        motivoID INT NOT NULL,
        ejecutivoID INT NOT NULL,
        proveedorID INT NOT NULL,
        competenciaID INT NOT NULL,
        idSociedad SMALLINT NOT NULL,
        sociedadCli6052 BIT NOT NULL,
        sociedadCli6012 BIT NOT NULL,
        tipoVisita TINYINT NOT NULL,                 -- 1 / 2 (significado por confirmar con negocio)
        estado BIT NOT NULL,
        posibilidadReversion BIT NOT NULL,
        flagMotivoCpsaa BIT NOT NULL,
        volumen DECIMAL(18,2) NOT NULL,
        volumenRestante DECIMAL(18,2) NOT NULL,
        precio DECIMAL(18,2) NOT NULL,
        fechaIngreso DATE NULL,
        fechaCreacion DATETIME2(3) NOT NULL,
        fechaModificacion DATETIME2(3) NULL,
        checkin DATETIME2(0) NULL,
        checkout DATETIME2(0) NULL,
        duracionMin AS (CASE WHEN checkout >= checkin THEN DATEDIFF(MINUTE, checkin, checkout) END) PERSISTED,
        checkinLatitud DECIMAL(10,7) NULL,
        checkinLongitud DECIMAL(10,7) NULL,
        checkoutLatitud DECIMAL(10,7) NULL,
        checkoutLongitud DECIMAL(10,7) NULL,
        acciones NVARCHAR(500) NULL,
        observacion NVARCHAR(500) NULL,
        FOREIGN KEY (tiempoID) REFERENCES Tiempo(id),
        FOREIGN KEY (oficinaID) REFERENCES Oficina(id),
        FOREIGN KEY (ubigeoID) REFERENCES Ubigeo(id),
        FOREIGN KEY (obraID) REFERENCES Obra(id),
        FOREIGN KEY (tipoEstructuraID) REFERENCES TipoEstructura(id),
        FOREIGN KEY (marcaID) REFERENCES Marca(id),
        FOREIGN KEY (unidadMedidaID) REFERENCES UnidadMedida(id),
        FOREIGN KEY (motivoID) REFERENCES Motivo(id),
        FOREIGN KEY (ejecutivoID) REFERENCES Ejecutivo(id),
        FOREIGN KEY (proveedorID) REFERENCES Proveedor(id),
        FOREIGN KEY (competenciaID) REFERENCES Competencia(id)
    )""",
]

# Miembro "DESCONOCIDO" (id = -1) de cada dimension: las filas sin llave caen aqui en vez de perderse.
DESCONOCIDOS = {
    "Oficina": "(id, codigo, nombre, idSociedad) VALUES (-1, -1, N'DESCONOCIDO', NULL)",
    "Ubigeo": "(id, codigo, codDepartamento, codProvincia, codDistrito, departamento, provincia, distrito) "
              "VALUES (-1, '000000', 0, 0, 0, N'DESCONOCIDO', N'DESCONOCIDO', N'DESCONOCIDO')",
    "Obra": "(id, codigo, idMaestroObra, tipoEjecucion) VALUES (-1, -1, NULL, N'DESCONOCIDO')",
    "TipoEstructura": "(id, codigo, nombre) VALUES (-1, -1, N'DESCONOCIDO')",
    "Marca": "(id, codigo, tipoMarca, codMarca, marca, marcaNormalizada, esMarcaPropia, codLinea, linea, "
             "codSublinea, sublinea, codProducto, producto) VALUES (-1, -1, N'DESCONOCIDO', NULL, N'DESCONOCIDO', "
             "N'DESCONOCIDO', 0, NULL, N'DESCONOCIDO', NULL, N'DESCONOCIDO', NULL, N'DESCONOCIDO')",
    "UnidadMedida": "(id, codigo, abreviatura) VALUES (-1, -1, N'DESCONOCIDO')",
    "Motivo": "(id, codigo, nombre) VALUES (-1, -1, N'DESCONOCIDO')",
    "Ejecutivo": "(id, codigo, ejecutivoSAP, nombre) VALUES (-1, -1, NULL, N'DESCONOCIDO')",
    "Proveedor": "(id, codigo, nombre) VALUES (-1, -1, N'DESCONOCIDO')",
    # Competencia: -1 = "NO APLICA" (0/0), el caso mas comun (~89% de las filas)
    "Competencia": "(id, codTipoCompetencia, tipoCompetencia, codMarcaCemento, marcaCemento) "
                   "VALUES (-1, 0, N'NO APLICA', 0, N'NO APLICA')",
}

INDICES = [
    "CREATE INDEX IX_fact_consumo_tiempo ON fact_consumo (tiempoID)",
    "CREATE INDEX IX_fact_consumo_obra ON fact_consumo (obraID)",
    "CREATE INDEX IX_fact_consumo_oficina ON fact_consumo (oficinaID)",
    "CREATE INDEX IX_fact_consumo_ubigeo ON fact_consumo (ubigeoID)",
    "CREATE INDEX IX_fact_consumo_marca ON fact_consumo (marcaID)",
    "CREATE INDEX IX_fact_consumo_visita ON fact_consumo (visitaID)",
]


# Vista plana (hechos + dimensiones) para que cualquier usuario consulte sin escribir JOINs.
# La usan el dashboard y la pestaña "Consultas SQL".
VISTA = """CREATE VIEW vw_consumo AS
    SELECT e.id AS id_encuesta, e.visitaID AS id_visita, ob.codigo AS id_obra, t.fecha, t.anio,
           CONCAT(t.anio, '-', RIGHT('0' + CAST(t.numMes AS VARCHAR(2)), 2)) AS mes,
           ofi.nombre AS oficina, u.codigo AS ubigeo, u.departamento, u.provincia, u.distrito,
           te.nombre AS tipo_estructura, ob.tipoEjecucion AS tipo_ejecucion,
           m.marca, m.marcaNormalizada AS marca_norm, m.esMarcaPropia AS es_propia,
           m.producto, m.linea, m.sublinea, um.abreviatura AS unidad,
           mo.nombre AS motivo, ej.nombre AS ejecutivo, pr.codigo AS proveedor_codigo,
           co.tipoCompetencia AS tipo_competencia, co.marcaCemento AS marca_competencia,
           e.volumen, e.volumenRestante AS volumen_restante, e.precio,
           e.tipoVisita AS tipo_visita, e.estado,
           e.checkinLatitud AS lat, e.checkinLongitud AS lon
    FROM fact_consumo e
    JOIN Tiempo         t   ON e.tiempoID         = t.id
    JOIN Oficina        ofi ON e.oficinaID        = ofi.id
    JOIN Ubigeo         u   ON e.ubigeoID         = u.id
    JOIN Obra           ob  ON e.obraID           = ob.id
    JOIN TipoEstructura te  ON e.tipoEstructuraID = te.id
    JOIN Marca          m   ON e.marcaID          = m.id
    JOIN UnidadMedida   um  ON e.unidadMedidaID   = um.id
    JOIN Motivo         mo  ON e.motivoID         = mo.id
    JOIN Ejecutivo      ej  ON e.ejecutivoID      = ej.id
    JOIN Proveedor      pr  ON e.proveedorID      = pr.id
    JOIN Competencia    co  ON e.competenciaID    = co.id"""


def create_datamart_schema(cursor):
    # Borra hechos primero y luego dimensiones; asi los cambios de esquema siempre se aplican
    cursor.execute("IF OBJECT_ID('vw_consumo', 'V') IS NOT NULL DROP VIEW vw_consumo")
    for t in TABLAS:
        cursor.execute(f"IF OBJECT_ID('{t}', 'U') IS NOT NULL DROP TABLE {t}")
    for sentencia in DDL:
        cursor.execute(sentencia)
    for tabla, valores in DESCONOCIDOS.items():
        cursor.execute(f"SET IDENTITY_INSERT {tabla} ON; INSERT INTO {tabla} {valores}; SET IDENTITY_INSERT {tabla} OFF;")
    cursor.execute(VISTA)
    cursor.commit()


# ----------------------------------------------------------------------------- LECTURA Y LIMPIEZA DEL ORIGEN
def leer_origen():
    print(f"Leyendo {RUTA_DATOS} ...")
    if RUTA_DATOS.lower().endswith(".csv"):
        return pd.read_csv(RUTA_DATOS, low_memory=False)
    return pd.read_excel(RUTA_DATOS, engine="openpyxl")


def preparar_origen(raw):
    df = raw.drop(columns=[c for c in DESCARTAR if c in raw.columns]).copy()

    enteros = ["iddia", "codcreadorom", "codencuestacensadores", "codigotipomarcalocal", "codvisita",
               "codigoobra", "idoficina", "cod_departamento", "cod_provincia", "cod_distrito", "idsociedad",
               "id_maestro_obra", "id_tipo_estructura", "codigomarcalocal", "codunidadmedidaisicom", "codlinea",
               "codsublinea", "codproducto", "codtipocompetencia_concreto", "codmotivoom", "ejecutivosapvisita",
               "codisicom_isicom_proveedor", "codisicommarcacemento", "tipo_de_visita"]
    for c in enteros:
        df[c] = pd.to_numeric(df[c], errors="coerce").astype("Int64")

    for c in ["fecha_ingreso_encuesta", "fecha_creacion_encuesta", "fecha_modificacion", "checkin", "checkout"]:
        df[c] = pd.to_datetime(df[c], errors="coerce")

    for c in ["posibilidadreversion_cen", "flagmotivocpsaa", "estado"]:
        df[c] = df[c].astype("boolean").fillna(False).astype(bool)
    for c in ["sociedad_cli_6052", "sociedad_cli_6012"]:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).ne(0)

    for c in ["volumencensadoresencuesta", "volumenrestantecensadoresencuesta", "preciocensadoresencuesta"]:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).round(2)

    textos = ["desaccionesom", "desobservacionom", "tipoejecucion_cen", "tipoestructura_cen", "destipomarca",
              "desmarca", "abrevunidadmedidaisicom", "deslinea", "dessublinea", "desproducto",
              "destipocompetencia_concreto", "desmotivoom", "descreadorom", "desisicommarcacemento"]
    for c in textos:
        df[c] = df[c].astype("string").str.strip().replace("", pd.NA)

    # Coordenadas: (0,0) no existe en Peru -> NULL (check-in y check-out por separado)
    for pref in ["checkin", "checkout"]:
        la, lo = f"{pref}_latitud", f"{pref}_longitud"
        malo = (df[la].fillna(0) == 0) & (df[lo].fillna(0) == 0)
        df[la] = df[la].mask(malo)
        df[lo] = df[lo].mask(malo)

    # Ubigeo de 6 digitos (dep + prov + dist). OJO: el codigo de distrito solo no es unico.
    df["ubigeo"] = ((df["cod_departamento"] * 10000 + df["cod_provincia"] * 100 + df["cod_distrito"])
                    .astype("Int64").astype(str).str.zfill(6))

    # Competencia: 0 = no aplica
    df["comp_tipo"] = df["codtipocompetencia_concreto"].fillna(0)
    df["comp_marca"] = df["codisicommarcacemento"].fillna(0)

    if df["codencuestacensadores"].duplicated().any():
        raise ValueError("Hay codencuestacensadores duplicados en el origen.")
    return df


def leer_nombres_oficina():
    if not os.path.exists(RUTA_OFICINAS):
        print(f"(sin {RUTA_OFICINAS}: las oficinas quedan como '{SIN_DESC}')")
        return pd.DataFrame(columns=["codigo", "nombre"])
    o = pd.read_csv(RUTA_OFICINAS)
    return o.rename(columns={"idoficina": "codigo", "desoficina": "nombre"})[["codigo", "nombre"]].drop_duplicates("codigo")


def leer_nombres_ubigeo():
    if not os.path.exists(RUTA_GEOJSON):
        print(f"(sin {RUTA_GEOJSON}: departamento/provincia/distrito quedan como '{SIN_DESC}')")
        return pd.DataFrame(columns=["codigo", "departamento", "provincia", "distrito"])
    with open(RUTA_GEOJSON, encoding="utf-8") as f:
        props = [x["properties"] for x in json.load(f)["features"]]
    g = pd.DataFrame(props).rename(columns={"IDDIST": "codigo", "NOMBDEP": "departamento",
                                             "NOMBPROV": "provincia", "NOMBDIST": "distrito"})
    return g[["codigo", "departamento", "provincia", "distrito"]].drop_duplicates("codigo")


# ----------------------------------------------------------------------------- CONSTRUCCION DE DIMENSIONES (en memoria)
def primero(df, clave, cols, ordenar_desc=None):
    """Una fila por clave; para cada columna toma el primer valor NO nulo (si una fila llega vacia, se usa la que si trae dato)."""
    d = df.dropna(subset=[clave] if isinstance(clave, str) else clave)
    if ordenar_desc:
        d = d.sort_values(ordenar_desc, ascending=False)
    return d.groupby(clave, as_index=False, sort=True)[cols].first()


def rellenar(s):
    return s.astype("string").fillna(SIN_DESC)


def construir_dimensiones(df):
    dims = {}

    # Tiempo: calendario completo
    f = pd.date_range(FECHA_INI, FECHA_FIN, freq="D")
    t = pd.DataFrame({"fecha": f})
    t["id"] = t.fecha.dt.year * 10000 + t.fecha.dt.month * 100 + t.fecha.dt.day
    t["anio"] = t.fecha.dt.year
    t["trimestre"] = t.fecha.dt.quarter
    t["numMes"] = t.fecha.dt.month
    t["mes"] = t.numMes.map(lambda m: MESES[m - 1])
    t["anioMes"] = t.anio * 100 + t.numMes
    t["dia"] = t.fecha.dt.day
    t["semanaISO"] = t.fecha.dt.isocalendar().week.astype(int)
    t["diaSemana"] = t.fecha.dt.dayofweek + 1
    t["nombreDia"] = t.diaSemana.map(lambda d: DIAS[d - 1])
    t["esFinDeSemana"] = t.diaSemana >= 6
    t["fecha"] = t.fecha.dt.date
    dims["Tiempo"] = t[["id", "fecha", "anio", "trimestre", "numMes", "mes", "anioMes", "dia",
                        "semanaISO", "diaSemana", "nombreDia", "esFinDeSemana"]]

    # Oficina (nombre desde oficinas.csv)
    o = primero(df, "idoficina", ["idsociedad"]).rename(columns={"idoficina": "codigo", "idsociedad": "idSociedad"})
    o = o.merge(leer_nombres_oficina(), on="codigo", how="left")
    o["nombre"] = rellenar(o["nombre"])
    o["codigo"] = o["codigo"].astype(int)
    dims["Oficina"] = o[["codigo", "nombre", "idSociedad"]]

    # Ubigeo (nombres desde el geojson publico)
    u = primero(df, "ubigeo", ["cod_departamento", "cod_provincia", "cod_distrito"])
    u = u.rename(columns={"ubigeo": "codigo", "cod_departamento": "codDepartamento",
                          "cod_provincia": "codProvincia", "cod_distrito": "codDistrito"})
    u = u.merge(leer_nombres_ubigeo(), on="codigo", how="left")
    for c in ["departamento", "provincia", "distrito"]:
        u[c] = rellenar(u[c])
    dims["Ubigeo"] = u[["codigo", "codDepartamento", "codProvincia", "codDistrito", "departamento", "provincia", "distrito"]]

    # Obra: atributos de la encuesta mas reciente de cada obra
    ob = primero(df, "codigoobra", ["id_maestro_obra", "tipoejecucion_cen"],
                 ordenar_desc=["fecha_creacion_encuesta", "codencuestacensadores"])
    ob = ob.rename(columns={"codigoobra": "codigo", "id_maestro_obra": "idMaestroObra", "tipoejecucion_cen": "tipoEjecucion"})
    ob["tipoEjecucion"] = ob["tipoEjecucion"].astype("string").fillna("DESCONOCIDO")
    dims["Obra"] = ob

    te = primero(df, "id_tipo_estructura", ["tipoestructura_cen"]).rename(
        columns={"id_tipo_estructura": "codigo", "tipoestructura_cen": "nombre"})
    te["nombre"] = rellenar(te["nombre"])
    dims["TipoEstructura"] = te

    # Marca (una fila por codigotipomarcalocal)
    m = primero(df, "codigotipomarcalocal", ["destipomarca", "codigomarcalocal", "desmarca", "codlinea", "deslinea",
                                             "codsublinea", "dessublinea", "codproducto", "desproducto"])
    m = m.rename(columns={"codigotipomarcalocal": "codigo", "destipomarca": "tipoMarca", "codigomarcalocal": "codMarca",
                          "desmarca": "marca", "codlinea": "codLinea", "deslinea": "linea", "codsublinea": "codSublinea",
                          "dessublinea": "sublinea", "codproducto": "codProducto", "desproducto": "producto"})
    for c in ["tipoMarca", "marca", "linea", "sublinea", "producto"]:
        m[c] = rellenar(m[c])
    m["marcaNormalizada"] = m["marca"].str.upper()
    m["esMarcaPropia"] = m["codMarca"].isin(MARCAS_PROPIAS)
    dims["Marca"] = m[["codigo", "tipoMarca", "codMarca", "marca", "marcaNormalizada", "esMarcaPropia", "codLinea",
                       "linea", "codSublinea", "sublinea", "codProducto", "producto"]]

    um = primero(df, "codunidadmedidaisicom", ["abrevunidadmedidaisicom"]).rename(
        columns={"codunidadmedidaisicom": "codigo", "abrevunidadmedidaisicom": "abreviatura"})
    um["abreviatura"] = rellenar(um["abreviatura"])
    dims["UnidadMedida"] = um

    mo = primero(df, "codmotivoom", ["desmotivoom"]).rename(columns={"codmotivoom": "codigo", "desmotivoom": "nombre"})
    mo["nombre"] = rellenar(mo["nombre"])
    dims["Motivo"] = mo

    ej = primero(df, "codcreadorom", ["ejecutivosapvisita", "descreadorom"]).rename(
        columns={"codcreadorom": "codigo", "ejecutivosapvisita": "ejecutivoSAP", "descreadorom": "nombre"})
    ej["nombre"] = rellenar(ej["nombre"])
    dims["Ejecutivo"] = ej

    pr = pd.DataFrame({"codigo": df["codisicom_isicom_proveedor"].dropna().unique()}).sort_values("codigo")
    pr["codigo"] = pr["codigo"].astype("Int64")
    pr["nombre"] = SIN_DESC
    dims["Proveedor"] = pr.reset_index(drop=True)

    # Competencia: (0,0) ya existe como miembro -1, no se inserta de nuevo
    co = primero(df, ["comp_tipo", "comp_marca"], ["destipocompetencia_concreto", "desisicommarcacemento"])
    co = co[~((co.comp_tipo == 0) & (co.comp_marca == 0))]
    co = co.rename(columns={"comp_tipo": "codTipoCompetencia", "comp_marca": "codMarcaCemento",
                            "destipocompetencia_concreto": "tipoCompetencia", "desisicommarcacemento": "marcaCemento"})
    co["tipoCompetencia"] = rellenar(co["tipoCompetencia"])
    co["marcaCemento"] = rellenar(co["marcaCemento"])
    dims["Competencia"] = co[["codTipoCompetencia", "tipoCompetencia", "codMarcaCemento", "marcaCemento"]]
    return dims


# Llaves de negocio: columna de la dimension -> columna del origen
LLAVES = {
    "Oficina": ({"codigo": "idoficina"}, "oficinaID"),
    "Ubigeo": ({"codigo": "ubigeo"}, "ubigeoID"),
    "Obra": ({"codigo": "codigoobra"}, "obraID"),
    "TipoEstructura": ({"codigo": "id_tipo_estructura"}, "tipoEstructuraID"),
    "Marca": ({"codigo": "codigotipomarcalocal"}, "marcaID"),
    "UnidadMedida": ({"codigo": "codunidadmedidaisicom"}, "unidadMedidaID"),
    "Motivo": ({"codigo": "codmotivoom"}, "motivoID"),
    "Ejecutivo": ({"codigo": "codcreadorom"}, "ejecutivoID"),
    "Proveedor": ({"codigo": "codisicom_isicom_proveedor"}, "proveedorID"),
    "Competencia": ({"codTipoCompetencia": "comp_tipo", "codMarcaCemento": "comp_marca"}, "competenciaID"),
}


# ----------------------------------------------------------------------------- UTILIDADES DE CARGA
def _nativo(v):
    if isinstance(v, np.generic):
        return v.item()
    if isinstance(v, pd.Timestamp):
        return v.to_pydatetime()
    return v


def a_filas(df):
    """DataFrame -> lista de tuplas con tipos nativos de Python y None en lugar de NaN/NA/NaT."""
    arr = df.astype(object).to_numpy()
    nulos = df.isna().to_numpy()
    return [tuple(None if n else _nativo(v) for v, n in zip(fila, mask)) for fila, mask in zip(arr, nulos)]


def insertar(cursor, tabla, df, lote=LOTE):
    cols = list(df.columns)
    sql = f"INSERT INTO {tabla} ({', '.join(cols)}) VALUES ({', '.join('?' * len(cols))})"
    filas = a_filas(df)
    for i in range(0, len(filas), lote):
        cursor.executemany(sql, filas[i:i + lote])


def cargar_dimension(cursor, tabla, df, claves):
    """Inserta la dimension y devuelve su tabla (id + llaves de negocio) para mapear la fact.
    Con cursor=None (dry-run) simula los ids."""
    if tabla == "Tiempo":
        if cursor is not None:
            insertar(cursor, tabla, df)
            cursor.commit()
        print(f"{len(df)} filas en {tabla}.")
        return df[["id"]]
    if cursor is None:
        mapa = df[claves].copy()
        mapa.insert(0, "id", range(1, len(df) + 1))
    else:
        insertar(cursor, tabla, df)
        cursor.commit()
        cursor.execute(f"SELECT id, {', '.join(claves)} FROM {tabla}")
        mapa = pd.DataFrame.from_records([tuple(r) for r in cursor.fetchall()], columns=["id"] + claves)
    print(f"{len(df)} filas en {tabla}.")
    return mapa


# ----------------------------------------------------------------------------- TABLA DE HECHOS
def construir_hechos(df, mapas):
    h = df.copy()
    for tabla, (llaves, col_id) in LLAVES.items():
        # nulos -> -1 (o 0 en competencia), para caer en el miembro DESCONOCIDO
        for _, col_origen in llaves.items():
            if col_origen not in ("ubigeo", "comp_tipo", "comp_marca"):
                h[col_origen] = h[col_origen].fillna(-1).astype("int64")
        m = mapas[tabla].rename(columns={"id": col_id})
        for col_dim, col_origen in llaves.items():
            if col_origen == "ubigeo":                 # llave de texto: mismo tipo en ambos lados
                h[col_origen] = h[col_origen].astype(object)
                m[col_dim] = m[col_dim].astype(object)
        n = len(h)
        h = h.merge(m, how="left", left_on=list(llaves.values()), right_on=list(llaves.keys()))
        if len(h) != n:
            raise ValueError(f"La dimension {tabla} tiene llaves duplicadas.")
        h = h.drop(columns=list(llaves.keys()))        # quita las llaves de la dimension para no chocar con la siguiente
        h[col_id] = h[col_id].fillna(-1).astype("int64")

    out = pd.DataFrame({
        "id": h["codencuestacensadores"].astype("int64"),
        "visitaID": h["codvisita"].fillna(0).astype("int64"),
        "tiempoID": h["iddia"].astype("int64"),
        "oficinaID": h["oficinaID"], "ubigeoID": h["ubigeoID"], "obraID": h["obraID"],
        "tipoEstructuraID": h["tipoEstructuraID"], "marcaID": h["marcaID"], "unidadMedidaID": h["unidadMedidaID"],
        "motivoID": h["motivoID"], "ejecutivoID": h["ejecutivoID"], "proveedorID": h["proveedorID"],
        "competenciaID": h["competenciaID"],
        "idSociedad": h["idsociedad"].fillna(0).astype("int64"),
        "sociedadCli6052": h["sociedad_cli_6052"], "sociedadCli6012": h["sociedad_cli_6012"],
        "tipoVisita": h["tipo_de_visita"].fillna(0).astype("int64"),
        "estado": h["estado"], "posibilidadReversion": h["posibilidadreversion_cen"],
        "flagMotivoCpsaa": h["flagmotivocpsaa"],
        "volumen": h["volumencensadoresencuesta"], "volumenRestante": h["volumenrestantecensadoresencuesta"],
        "precio": h["preciocensadoresencuesta"],
        "fechaIngreso": h["fecha_ingreso_encuesta"].dt.date,
        "fechaCreacion": h["fecha_creacion_encuesta"],
        "fechaModificacion": h["fecha_modificacion"],
        "checkin": h["checkin"], "checkout": h["checkout"],
        "checkinLatitud": h["checkin_latitud"].round(7), "checkinLongitud": h["checkin_longitud"].round(7),
        "checkoutLatitud": h["checkout_latitud"].round(7), "checkoutLongitud": h["checkout_longitud"].round(7),
        "acciones": h["desaccionesom"].str.slice(0, 500), "observacion": h["desobservacionom"].str.slice(0, 500),
    })
    # fechaCreacion es NOT NULL: si faltara, se usa el dia de la encuesta
    falta = out["fechaCreacion"].isna()
    if falta.any():
        out.loc[falta, "fechaCreacion"] = pd.to_datetime(out.loc[falta, "tiempoID"].astype(str), format="%Y%m%d")
    # los hechos no pueden apuntar a fechas fuera de Tiempo
    fuera = ~out["tiempoID"].between(FECHA_INI.year * 10000 + 101, FECHA_FIN.year * 10000 + 1231)
    if fuera.any():
        raise ValueError(f"{int(fuera.sum())} encuestas con iddia fuera del rango de Tiempo ({FECHA_INI} a {FECHA_FIN}).")
    return out


# ----------------------------------------------------------------------------- VALIDACION
def validar(cursor, hechos, dims):
    print("\n--- Validación ---")
    esperado = (len(hechos), round(float(hechos.volumen.sum()), 2),
                round(float(hechos.volumenRestante.sum()), 2), round(float(hechos.precio.sum()), 2))
    print(f"Origen  : filas={esperado[0]}, volumen={esperado[1]:,.2f}, restante={esperado[2]:,.2f}, precio={esperado[3]:,.2f}")
    if cursor is None:
        print("(dry-run: no se consulta SQL Server)")
    else:
        cursor.execute("SELECT COUNT(*), SUM(volumen), SUM(volumenRestante), SUM(precio) FROM fact_consumo")
        r = cursor.fetchone()
        real = (r[0], round(float(r[1]), 2), round(float(r[2]), 2), round(float(r[3]), 2))
        print(f"Destino : filas={real[0]}, volumen={real[1]:,.2f}, restante={real[2]:,.2f}, precio={real[3]:,.2f}")
        print("✅ Totales coinciden" if real == esperado else "❌ LOS TOTALES NO COINCIDEN, revisar")

    print("Filas en DESCONOCIDO (-1) por dimensión:")
    for col in ["oficinaID", "ubigeoID", "obraID", "tipoEstructuraID", "marcaID", "unidadMedidaID", "motivoID",
                "ejecutivoID", "proveedorID", "competenciaID"]:
        print(f"  {col:18s} {int((hechos[col] == -1).sum())}")
    print("Filas por dimensión (sin el miembro -1):", {k: len(v) for k, v in dims.items()})


# ----------------------------------------------------------------------------- PROGRAMA PRINCIPAL
def main():
    dry = "--dry-run" in sys.argv

    df = preparar_origen(leer_origen())
    print(f"{len(df)} encuestas leídas ({df.iddia.min()} a {df.iddia.max()}).")
    dims = construir_dimensiones(df)

    if dry:
        cursor = None
        conn = None
    else:
        if not crear_base_si_no_existe():
            sys.exit(1)
        conn = get_connection_destino()
        if not conn:
            sys.exit(1)
        cursor = conn.cursor()
        cursor.fast_executemany = True           # ~100 veces más rápido que insertar fila por fila
        create_datamart_schema(cursor)
        print(" Esquema completo del Data Mart creado.")

    # ----------------------------------------------------------- Cargar dimensiones
    mapas = {}
    print("Cargando Tiempo...")
    cargar_dimension(cursor, "Tiempo", dims["Tiempo"], ["id"])
    for tabla, (llaves, _) in LLAVES.items():
        print(f"Cargando {tabla}...")
        mapas[tabla] = cargar_dimension(cursor, tabla, dims[tabla], list(llaves.keys()))

    # ----------------------------------------------------------- Tabla de hechos: fact_consumo
    print("Cargando fact_consumo...")
    hechos = construir_hechos(df, mapas)
    if cursor is not None:
        insertar(cursor, "fact_consumo", hechos)
        conn.commit()
        for sentencia in INDICES:
            cursor.execute(sentencia)
        conn.commit()
    print(f"{len(hechos)} encuestas cargadas.")

    validar(cursor, hechos, dims)
    if conn:
        conn.close()
    print(" ETL completado exitosamente.")


if __name__ == "__main__":
    main()


Leyendo data_origen_1.xlsx ...
248286 encuestas leídas (20230102 a 20251231).
✓ Conexión a destino exitosa
 Esquema completo del Data Mart creado.
Cargando Tiempo...
3287 filas en Tiempo.
Cargando Oficina...
14 filas en Oficina.
Cargando Ubigeo...
351 filas en Ubigeo.
Cargando Obra...
77519 filas en Obra.
Cargando TipoEstructura...
14 filas en TipoEstructura.
Cargando Marca...
107 filas en Marca.
Cargando UnidadMedida...
9 filas en UnidadMedida.
Cargando Motivo...
13 filas en Motivo.
Cargando Ejecutivo...
101 filas en Ejecutivo.
Cargando Proveedor...
2563 filas en Proveedor.
Cargando Competencia...
17 filas en Competencia.
Cargando fact_consumo...
248286 encuestas cargadas.

--- Validación ---
Origen  : filas=248286, volumen=21,952,777.34, restante=45,930,444.93, precio=17,161,150.64
Destino : filas=248286, volumen=21,952,777.34, restante=45,930,444.93, precio=17,161,150.64
✅ Totales coinciden
Filas en DESCONOCIDO (-1) por dimensión:
  oficinaID          0
  ubigeoID           0
  obra

In [3]:
!pip install dash

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/9.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.3 MB ? eta -:--:--
   -- ------------------------------------- 0.5/9.3 MB 2.5 MB/s eta 0:00:04
   ------ --------------------------------- 1.6/9.3 MB 4.4 MB/s eta 0:00:02
   ---------------------- ----------------- 5.2/9.3 MB 10.3 MB/s eta 0:00:01
   ---------------------------------------  9.2/9.3 MB 12.5 MB/s eta 0:00:01
   ---------------------------------------- 9.3/9.3 MB 11.3 MB/s  0:00:01

   -------------------------- ------------- 2/3 [dash]
   -------------------------- ------------- 2/3 [dash]
   -------------------------- ------------- 2/3 [dash]
   -------------------------- ------------- 2/3 [dash]
   -------------------------- ------------- 2/3 [dash]
   -------------------------- ------------- 2/3 [dash]
   -------------------------- ------------- 2/3 [dash]
   ---------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [5]:
# -*- coding: utf-8 -*-
"""
=====================================================================
  DASHBOARD: ¿POR QUÉ CAE EL MARKET SHARE DEL CEMENTO CEMEX?
  Dash + Plotly  ·  Datos desde SQL Server  ·  Colores Cemex
=====================================================================
  Basado en la plantilla de la clase (dash_controles_y_graficos.py):
  misma estructura (1 Configuración, 2 Carga de datos, 3 Funciones de
  gráficos, 4 Layout, 5 Callbacks, 6 Iniciar), mismos nombres de ayudas
  (ESTILO_TARJETA, ESTILO_GRAFICO, tarjeta_kpi, formato) y la misma idea:
  una función = un gráfico, y los callbacks solo los conectan a los filtros.

  PROBLEMA
    En los últimos meses la gerencia observa una caída sostenida del
    market share del cemento Cemex.

  OBJETIVOS ESPECÍFICOS (cada uno tiene su pestaña)
    1. KPI del estado del market share y cómo evolucionan     -> pestaña 1
    2. ¿En qué zonas, segmentos y periodos se concentra la caída? -> pestaña 2
    3. ¿Quién captura el share que se pierde? ¿Influye el precio? -> pestaña 3
    4. ¿Cómo mejorar el share? Propuesta, simulador e indicadores -> pestaña 4
    5. Consultas SQL libres sobre el datamart                  -> pestaña 5

  DEFINICIÓN DE MARKET SHARE
    Share de volumen = bolsas de cemento Cemex / bolsas de cemento de todas
    las marcas, en las obras censadas (autoconstrucción). Es el "share de lo
    censado", no el del mercado total del país. Se analiza TODO el cemento
    (todas las marcas); la marca propia la define el datamart
    (Marca.esMarcaPropia).

  ALCANCE REAL DE LOS DATOS (importante)
    - Periodo disponible: 01/2023 a 12/2025 (no hay 2021-2022 ni 2026).
    - Es un censo de obras, no despachos facturados.
    - No existe un campo de "canal de venta" (el proveedor viene vacío).
      El análisis de canal se aproxima con tipo de ejecución de la obra,
      etapa de obra y sociedad (6012 / 6052). Ver pestaña 4.

  CÓMO EJECUTAR
      1) Crear y cargar el datamart (una vez):
             python codigo_completo_datamart_autoconstruccion.py
      2) pip install dash plotly pandas numpy openpyxl pyodbc sqlglot
         (y el "ODBC Driver 17 (o 18) for SQL Server" de Microsoft)
      3) Conexión por variables de entorno (opcionales):
             SQL_SERVER (localhost)  SQL_DATABASE (DatamartAutoconstruccion)
             SQL_UID (sa; vacío = autenticación de Windows)  SQL_PWD
             SQL_UID_LECTURA / SQL_PWD_LECTURA: usuario de SOLO LECTURA
             para la pestaña "Consultas SQL" (recomendado).
      4) python dashboard_marketshare_cemex.py
         Abrir: http://localhost:8054

  ARCHIVOS (misma carpeta que este script)
      peru_distrital_simple.geojson   polígonos para el mapa
      assets/logo_cemex.png           logo del encabezado (opcional)
=====================================================================
"""

# ---------------------------------------------------------------------
# IMPORTACIONES
# ---------------------------------------------------------------------
import datetime
import decimal
import json
import os
import re
import sys
import threading
import time
import uuid
from collections import OrderedDict
from functools import lru_cache
from types import SimpleNamespace

import numpy as np
import pandas as pd

import dash
from dash import dcc, html, dash_table, Input, Output, State
from dash.dash_table.Format import Format, Group, Scheme, Sign

import plotly.express as px
import plotly.graph_objects as go

try:                                   # sqlglot valida las consultas de la pestaña SQL; sin él la validación es más básica
    import sqlglot
    from sqlglot import exp
except ImportError:
    sqlglot = None


# =====================================================================
# 1. CONFIGURACIÓN
# =====================================================================
# Carpeta donde están el geojson y assets/. En Jupyter no existe __file__: se usa la carpeta de trabajo actual.
# (Si el geojson está en otra carpeta, escriba aquí su ruta, por ejemplo CARPETA = r'C:\proyecto\dashboard')
CARPETA = os.path.dirname(os.path.abspath(__file__)) if '__file__' in globals() else os.getcwd()

# ---- Conexión a SQL Server (mismos nombres de variable que el script del datamart) ----
SERVIDOR = os.getenv('SQL_SERVER', 'localhost')
BASE_DATOS = os.getenv('SQL_DATABASE', 'DatamartAutoconstruccion')
USUARIO = os.getenv('SQL_UID', 'sa')                       # vacío = autenticación de Windows
CLAVE = os.getenv('SQL_PWD', '1234567890')                 # recomendado: definir SQL_PWD como variable de entorno
USUARIO_LECTURA = os.getenv('SQL_UID_LECTURA', USUARIO)    # usuario de solo lectura para la consola SQL
CLAVE_LECTURA = os.getenv('SQL_PWD_LECTURA', CLAVE)
DRIVER_ODBC = os.getenv('SQL_DRIVER', '')                  # vacío = se detecta el más nuevo instalado

# Qué se analiza
PRODUCTO = 'Cemento'          # todas las marcas de cemento (la "línea cemento" completa)
UNIDAD = 'BLS'                # bolsas: es el 99.99 % del cemento censado (las otras unidades no se pueden sumar)
SOLO_ACTIVAS = True           # excluye encuestas dadas de baja (estado = 0)

# Consulta que devuelve UNA tabla plana: la tabla de hechos unida con sus dimensiones.
# Los alias (AS) son los nombres de columna que usa todo el script. Los dos "?" se llenan con PRODUCTO y UNIDAD.
CONSULTA_SQL = """
    SELECT e.id AS id_encuesta, ob.codigo AS id_obra, t.fecha,
           ofi.nombre AS oficina, u.codigo AS ubigeo, u.departamento, u.provincia, u.distrito,
           te.nombre AS tipo_estructura, ob.tipoEjecucion AS tipo_ejecucion, e.idSociedad AS sociedad,
           mo.nombre AS motivo, m.marca, m.marcaNormalizada AS marca_norm, m.esMarcaPropia AS es_propia,
           e.volumen, e.precio, e.checkinLatitud AS lat, e.checkinLongitud AS lon
    FROM fact_consumo e
    JOIN Tiempo         t   ON e.tiempoID         = t.id
    JOIN Oficina        ofi ON e.oficinaID        = ofi.id
    JOIN Ubigeo         u   ON e.ubigeoID         = u.id
    JOIN Obra           ob  ON e.obraID           = ob.id
    JOIN TipoEstructura te  ON e.tipoEstructuraID = te.id
    JOIN Marca          m   ON e.marcaID          = m.id
    JOIN UnidadMedida   um  ON e.unidadMedidaID   = um.id
    JOIN Motivo         mo  ON e.motivoID         = mo.id
    WHERE m.producto = ? AND um.abreviatura = ?
"""
if SOLO_ACTIVAS:
    CONSULTA_SQL += "      AND e.estado = 1\n"

RUTA_GEOJSON = os.path.join(CARPETA, 'peru_distrital_simple.geojson')
RUTA_ASSETS = os.path.join(CARPETA, 'assets')
ARCHIVO_LOGO = 'logo_cemex.png'

# ---- Análisis ----
VENTANA_DEF = 6               # "últimos N meses" por defecto
UMBRAL_ALERTA = 1.0           # pp por debajo de la meta -> "Alerta"
UMBRAL_CRITICO = 3.0          # pp por debajo de la meta -> "Crítico"
UMBRAL_OFICINA = 3.0          # una oficina está "en alerta" si su share cae al menos estos pp
MIN_VOLUMEN_DEF = 250         # bolsas mínimas por distrito (en cada periodo) para el mapa y la tabla
MESES_HEATMAP = 18            # meses que muestra el mapa de calor oficina x mes

# ---- Consola de consultas SQL ----
MAX_FILAS = 50000             # filas máximas que devuelve una consulta
MAX_FILAS_TABLA = 5000        # filas que se muestran en pantalla (la descarga trae hasta MAX_FILAS)
SQL_TIMEOUT = 60              # segundos máximos por consulta

TITULO = 'Market share de cemento Cemex'
NOMBRE_VALOR = 'bolsas'       # cómo se llama la medida en pantalla
# ---- Acceso web (para compartir el dashboard por un túnel como ngrok) ----
# Si define DASH_USER y DASH_PASS, el navegador pide usuario y contraseña antes de mostrar nada. Vacío = sin contraseña.
USUARIO_WEB = os.getenv('DASH_USER', '')
CLAVE_WEB = os.getenv('DASH_PASS', '')
PUERTO = 8054                 # 8050 base · 8052 plantilla · 8053 dashboard anterior
DEBUG = False                 # True recarga al guardar, pero lee la base de datos dos veces

# ---- Colores Cemex (muestreados del logo) ----
AZUL = '#0103b1'              # azul Cemex: marca propia, encabezados, botones, valores clave
ROJO = '#fa2232'              # rojo Cemex: acento (franja, pestaña activa, caídas, líneas de meta)
ROJO_TEXTO = '#c8101e'        # rojo más oscuro para TEXTO sobre blanco (el del logo no llega a contraste 4.5:1)
VERDE_TEXTO = '#0a7d52'       # estado "En meta"
AMBAR_TEXTO = '#9a6700'       # estado "Alerta"
FONDO = '#f3f4fa'
BORDE = '#d9dbec'
AZUL_MEDIO = '#6a6dd4'
AZUL_CLARO = '#b9bbe6'
# Colores fijos por marca (elegidos para distinguirse del azul Cemex y entre sí, también con daltonismo)
COLORES_COMPETENCIA = ['#1baf7a', '#eb6834', '#e87ba4', '#eda100']      # verde, naranja, rosa, ámbar
COLOR_OTRAS = '#9aa0a6'
TINTA = '#14152b'
TINTA_SEC = '#4b4f66'
TINTA_SUAVE = '#6b6f86'
ESCALA_SHARE = [[0.0, '#eceefb'], [0.5, AZUL_MEDIO], [1.0, AZUL]]       # secuencial: claro -> azul Cemex
ESCALA_DELTA = [[0.0, ROJO], [0.5, '#efeff7'], [1.0, AZUL]]             # divergente: cae (rojo) / sube (azul)

# Ubigeos que el geojson no trae (departamento, provincia, distrito). 200115 = Veintiséis de Octubre (Piura); verificar.
NOMBRES_MANUALES = {'200115': ('PIURA', 'PIURA', 'VEINTISEIS DE OCTUBRE')}
# Textos que el datamart usa cuando falta el dato; en pantalla se muestran como "Sin dato"
VALORES_FALTANTES = {'DESCONOCIDO', 'SIN DESCRIPCION', 'NO APLICA'}
MESES_ES = ['ene', 'feb', 'mar', 'abr', 'may', 'jun', 'jul', 'ago', 'sep', 'oct', 'nov', 'dic']

# Segmentos por los que se puede ver la caída (pestaña 2). "Canal" se aproxima con estos campos.
DIMENSIONES = {'Oficina': 'oficina', 'Departamento': 'departamento',
               'Tipo de ejecución de la obra': 'tipo_ejecucion', 'Etapa de obra': 'tipo_estructura',
               'Sociedad (6012 / 6052)': 'sociedad'}


# =====================================================================
# 2. CARGA DE DATOS (SQL SERVER)
# =====================================================================
class ErrorDeConexion(Exception):
    """Error con un mensaje en español, listo para mostrar al usuario."""


def _valor_odbc(v):
    """Encierra un valor entre llaves para que ';' o '}' en la clave no rompan la cadena de conexión."""
    return '{' + str(v).replace('}', '}}') + '}'


def _elegir_driver():
    import pyodbc
    instalados = pyodbc.drivers()
    if DRIVER_ODBC:
        return DRIVER_ODBC
    for nombre in ('ODBC Driver 18 for SQL Server', 'ODBC Driver 17 for SQL Server', 'ODBC Driver 13 for SQL Server',
                   'SQL Server Native Client 11.0', 'SQL Server'):
        if nombre in instalados:
            return nombre
    raise ErrorDeConexion('No se encontró ningún driver ODBC de SQL Server en este equipo. '
                          'Instale "ODBC Driver 17 for SQL Server" (o 18) desde Microsoft. '
                          f'Drivers detectados: {instalados or "ninguno"}.')


def cadena_conexion(solo_lectura=False):
    uid, pwd = (USUARIO_LECTURA, CLAVE_LECTURA) if solo_lectura else (USUARIO, CLAVE)
    driver = _elegir_driver()
    partes = [f'DRIVER={_valor_odbc(driver)}', f'SERVER={_valor_odbc(SERVIDOR)}', f'DATABASE={_valor_odbc(BASE_DATOS)}']
    if uid:
        partes += [f'UID={_valor_odbc(uid)}', f'PWD={_valor_odbc(pwd)}']
    else:
        partes.append('Trusted_Connection=yes')
    if driver.startswith('ODBC Driver 1'):
        partes.append('TrustServerCertificate=yes')        # el driver 18 exige cifrado; los servidores locales usan certificado propio
    return ';'.join(partes)


def conectar(solo_lectura=False, timeout_login=15):
    try:
        import pyodbc
    except ImportError:
        raise ErrorDeConexion('Falta la librería pyodbc. Instálela con:  pip install pyodbc')
    try:
        return pyodbc.connect(cadena_conexion(solo_lectura), timeout=timeout_login, autocommit=True)
    except pyodbc.Error as e:
        raise ErrorDeConexion(explicar_error_sql(e, solo_lectura)) from e


def limpiar_error_sql(e):
    """Deja solo el mensaje de SQL Server, sin los prefijos del driver ODBC."""
    texto = str(e.args[-1]) if e.args else str(e)
    m = re.search(r'\[SQL Server\](.*?)(?:\s*\(\d+\))*\s*\(SQL\w+\)', texto, re.S)
    if m:
        return m.group(1).strip()
    texto = re.sub(r'^(\[[^\]]*\]\s*)+', '', texto)                  # quita "[08001] [Microsoft][ODBC Driver ...]"
    return re.sub(r'\s*\(SQL\w+\)\s*$', '', texto).strip()


def explicar_error_sql(e, solo_lectura=False):
    estado = str(e.args[0]) if e.args else ''
    detalle = limpiar_error_sql(e)
    usuario = f'de lectura "{USUARIO_LECTURA}"' if solo_lectura else f'"{USUARIO or "(Windows)"}"'
    if estado in ('28000', '28P01') or 'Login failed' in detalle:
        return (f'SQL Server rechazó el inicio de sesión del usuario {usuario}. Revise SQL_UID / SQL_PWD '
                f'(o SQL_UID_LECTURA / SQL_PWD_LECTURA). Detalle: {detalle}')
    if estado in ('08001', '08S01', 'HYT00') and 'Invalid object' not in detalle:
        return (f'No se pudo conectar al servidor "{SERVIDOR}". Verifique el nombre (variable SQL_SERVER), que SQL Server '
                f'esté encendido y que acepte conexiones TCP/IP. Detalle: {detalle}')
    if estado == '42000' and 'Cannot open database' in detalle:
        return (f'No se puede abrir la base "{BASE_DATOS}": no existe o el usuario {usuario} no tiene permiso. '
                f'Si aún no la creó, ejecute primero codigo_completo_datamart_autoconstruccion.py. Detalle: {detalle}')
    if estado == '42S02' or 'Invalid object name' in detalle:
        return (f'Falta una tabla del datamart en "{BASE_DATOS}". Ejecute primero '
                f'codigo_completo_datamart_autoconstruccion.py para crearlo y cargarlo. Detalle: {detalle}')
    if estado == '42S22' or 'Invalid column name' in detalle:
        return (f'El datamart de "{BASE_DATOS}" no tiene una columna que el dashboard espera. Vuelva a ejecutar '
                f'codigo_completo_datamart_autoconstruccion.py (versión con fact_consumo). Detalle: {detalle}')
    return detalle or f'{type(e).__name__}'


def _numero(serie):
    """Decimal (lo que entrega SQL Server para DECIMAL) -> float, para poder operar con numpy/pandas."""
    return pd.to_numeric(serie.map(lambda v: float(v) if isinstance(v, decimal.Decimal) else v), errors='coerce')


def cargar_desde_sql():
    conn = conectar()
    try:
        cur = conn.cursor()
        cur.execute(CONSULTA_SQL, (PRODUCTO, UNIDAD))
        columnas = [c[0] for c in cur.description]
        filas = [tuple(f) for f in cur.fetchall()]
    except Exception as e:
        import pyodbc
        if isinstance(e, pyodbc.Error):
            raise ErrorDeConexion(explicar_error_sql(e)) from e
        raise
    finally:
        conn.close()
    return pd.DataFrame.from_records(filas, columns=columnas)


def leer_geojson():
    if not os.path.exists(RUTA_GEOJSON):
        print(f'(sin {os.path.basename(RUTA_GEOJSON)}: el mapa no estará disponible)')
        return {}
    with open(RUTA_GEOJSON, encoding='utf-8') as f:
        feats = json.load(f)['features']
    return {x['properties']['IDDIST']: x for x in feats if x.get('geometry')}     # algunos distritos vienen sin geometría


def preparar(d):
    d = d.copy()
    d['fecha'] = pd.to_datetime(d['fecha'])
    for c in ['volumen', 'precio', 'lat', 'lon']:
        d[c] = _numero(d[c])
    d['volumen'] = d['volumen'].fillna(0)
    d['es_propia'] = d['es_propia'].astype(bool)
    d['sociedad'] = d['sociedad'].map(lambda v: 'Sin dato' if pd.isna(v) else str(int(v)))
    for c in ['oficina', 'departamento', 'provincia', 'distrito', 'tipo_estructura', 'tipo_ejecucion',
              'marca', 'marca_norm', 'motivo']:
        s = d[c].astype('object')
        falta = s.isna() | s.astype(str).str.strip().str.upper().isin(VALORES_FALTANTES) | (s.astype(str).str.strip() == '')
        d[c] = s.where(~falta, 'Sin dato')
    d['ubigeo'] = d['ubigeo'].astype(str).str.strip().str.zfill(6)
    for ub, (dep, pro, dis) in NOMBRES_MANUALES.items():
        falta = (d['ubigeo'] == ub) & (d['distrito'] == 'Sin dato')
        d.loc[falta, ['departamento', 'provincia', 'distrito']] = [dep, pro, dis]
    sin_nombre = d['distrito'] == 'Sin dato'
    d.loc[sin_nombre, 'distrito'] = 'Ubigeo ' + d.loc[sin_nombre, 'ubigeo']
    # 'Holcim' y 'HOLCIM' son la misma marca: se une por marca_norm y se muestra el nombre más frecuente
    nombre = d.groupby('marca_norm')['marca'].agg(lambda s: s.mode().iat[0])
    d['marca'] = d['marca_norm'].map(nombre)
    d['mes'] = d['fecha'].dt.strftime('%Y-%m')
    malo = (d.lat.fillna(0) == 0) | (d.lon.fillna(0) == 0) | (d.lat > -0.5)      # coordenadas inválidas
    d.loc[malo, ['lat', 'lon']] = np.nan
    return d.sort_values('fecha').reset_index(drop=True)


GEO = leer_geojson()
try:
    print(f'Conectando a SQL Server ({SERVIDOR} / {BASE_DATOS})...')
    df = preparar(cargar_desde_sql())
except ErrorDeConexion as _e:
    print(f'\n✖ No se pudieron leer los datos de SQL Server.\n  {_e}\n')
    sys.exit(1)
if df.empty:
    print(f'\n✖ La consulta no devolvió filas para {PRODUCTO} en {UNIDAD}. ¿Cargó el datamart? '
          f'(codigo_completo_datamart_autoconstruccion.py)\n')
    sys.exit(1)
print(f'✓ {len(df):,} filas leídas desde SQL Server')

# ---- listas y constantes para los controles (valores sin repetir, ordenados) ----
FECHA_MIN, FECHA_MAX = df.fecha.min(), df.fecha.max()
MES_MIN, MES_FIN = df.mes.min(), df.mes.max()
OPC_DEPTO = sorted(df.departamento.unique())
OPC_OFICINA = sorted(df.oficina.unique())
OPC_ETAPA = sorted(df.tipo_estructura.unique())
OPC_EJEC = sorted(df.tipo_ejecucion.unique())
N_OFICINAS = df.oficina.nunique()

# Centroides de distrito (mediana de las coordenadas de check-in) para dibujar distritos sin polígono
CENTROIDES = df.dropna(subset=['lat', 'lon']).groupby('ubigeo').agg(lat=('lat', 'median'), lon=('lon', 'median'))
NOMBRES_UBIGEO_DF = df.groupby('ubigeo', as_index=False)[['departamento', 'provincia', 'distrito']].first()

# Color fijo por marca: la propia en azul Cemex y las 4 competidoras con más volumen en colores distintos
MARCAS_PROPIAS = df[df.es_propia].groupby('marca').volumen.sum().sort_values(ascending=False).index.tolist()
TOP_COMPETENCIA = df[~df.es_propia].groupby('marca').volumen.sum().sort_values(ascending=False).head(4).index.tolist()
COLOR_MARCA = {m: [AZUL, AZUL_MEDIO, AZUL_CLARO][i % 3] for i, m in enumerate(MARCAS_PROPIAS)}
COLOR_MARCA.update({m: COLORES_COMPETENCIA[i] for i, m in enumerate(TOP_COMPETENCIA)})


def color_marca(m):
    return COLOR_MARCA.get(m, COLOR_OTRAS)


# =====================================================================
# 3. FUNCIONES DE CÁLCULO
# =====================================================================
def mes_mas(mes, k):
    """Suma k meses a un texto 'AAAA-MM'. mes_mas('2025-12', -5) -> '2025-07'."""
    return str(pd.Period(mes, 'M') + k)


def etiqueta_periodo(ini, fin):
    a, b = pd.Period(ini, 'M'), pd.Period(fin, 'M')
    f = lambda p: f'{MESES_ES[p.month - 1]} {p.year}'
    return f(a) if a == b else f'{f(a)} – {f(b)}'


def formato(valor, decimales=0):
    """Número con separador de miles: 1234567.8 -> '1,234,568'."""
    return '—' if valor is None or (isinstance(valor, float) and np.isnan(valor)) else f'{valor:,.{decimales}f}'


def pp(x, dec=1):
    """Diferencia en puntos porcentuales con signo: -7.04 -> '-7.0 pp'."""
    return '—' if x is None or np.isnan(x) else f'{x:+.{dec}f} pp'


def _ok(x):
    return x is not None and not (isinstance(x, float) and np.isnan(x))


def share_vol(d):
    """Share de volumen de la marca propia (%). NaN si no hay datos."""
    if d is None or d.empty:
        return np.nan
    t = d.volumen.sum()
    return d.loc[d.es_propia, 'volumen'].sum() / t * 100 if t > 0 else np.nan


def share_obras(d):
    """Penetración: % de las obras censadas donde aparece la marca propia."""
    if d is None or d.empty:
        return np.nan
    return d.loc[d.es_propia, 'id_obra'].nunique() / d.id_obra.nunique() * 100


def vol_propio(d):
    return 0.0 if d is None or d.empty else float(d.loc[d.es_propia, 'volumen'].sum())


def vol_comp(d):
    return 0.0 if d is None or d.empty else float(d.loc[~d.es_propia, 'volumen'].sum())


def crecimiento(actual, previo):
    """Variación porcentual; NaN si no hay base."""
    return (actual / previo - 1) * 100 if previo and previo > 0 else np.nan


def precio_ponderado(d):
    """Precio promedio ponderado por volumen. Se recortan los precios fuera del rango p1-p99 (errores de digitación)."""
    if d is None or d.empty:
        return np.nan
    x = d[(d.precio > 0) & (d.volumen > 0)]
    if len(x) < 10:
        return np.nan
    lo, hi = x.precio.quantile([0.01, 0.99])
    x = x[x.precio.between(lo, hi)]
    return (x.precio * x.volumen).sum() / x.volumen.sum() if x.volumen.sum() > 0 else np.nan


def indice_precio(d):
    """Precio propio / precio de la competencia x 100 (100 = igual). >100: Cemex es más caro."""
    if d is None or d.empty:
        return np.nan
    a, b = precio_ponderado(d[d.es_propia]), precio_ponderado(d[~d.es_propia])
    return a / b * 100 if _ok(a) and _ok(b) and b > 0 else np.nan


MESES_TODOS = [str(p) for p in pd.period_range(MES_MIN, MES_FIN, freq='M')]


def serie_mensual(d):
    """Una fila por mes: volumen total / propio / competencia, share y media móvil de 3 meses."""
    g = d.assign(propio=np.where(d.es_propia, d.volumen, 0.0)).groupby('mes').agg(vol=('volumen', 'sum'), propio=('propio', 'sum'))
    g = g.reindex(MESES_TODOS)
    g['comp'] = g.vol - g.propio
    g['share'] = g.propio / g.vol * 100
    g['share_ma3'] = g.propio.rolling(3, min_periods=3).sum() / g.vol.rolling(3, min_periods=3).sum() * 100
    return g


COLUMNAS_SEG = ['vol_ult', 'vol_ref', 'prop_ult', 'prop_ref', 'w_ult', 'w_ref', 's_ult', 's_ref',
                'delta', 'mix', 'tasa', 'aporte', 'recuperable']


def tabla_segmento(ult, ref, col):
    """Compara dos periodos por segmento (oficina, distrito, etapa...) y DESCOMPONE el cambio del share total:

        cambio total = SUMA( efecto mezcla ) + SUMA( efecto tasa )
        efecto tasa   = peso_actual x (share_actual - share_anterior)   -> el share cayó DENTRO del segmento
        efecto mezcla = (peso_actual - peso_anterior) x (share_anterior - share_total_anterior)
                                                                        -> cambió cuánto pesa cada segmento
    La suma da EXACTAMENTE el cambio del share total. 'recuperable' = bolsas que habría que volver a ganar
    en el segmento para recuperar su share anterior."""
    if ult is None or ref is None or ult.empty or ref.empty:
        return pd.DataFrame(columns=COLUMNAS_SEG)
    vu, vr = ult.groupby(col).volumen.sum(), ref.groupby(col).volumen.sum()
    pu = ult[ult.es_propia].groupby(col).volumen.sum()
    pr = ref[ref.es_propia].groupby(col).volumen.sum()
    t = pd.DataFrame({'vol_ult': vu, 'vol_ref': vr, 'prop_ult': pu, 'prop_ref': pr}).fillna(0.0)
    S_u, S_r = t.prop_ult.sum() / t.vol_ult.sum() * 100, t.prop_ref.sum() / t.vol_ref.sum() * 100
    t['w_ult'], t['w_ref'] = t.vol_ult / t.vol_ult.sum(), t.vol_ref / t.vol_ref.sum()
    t['s_ult'] = np.where(t.vol_ult > 0, t.prop_ult / t.vol_ult.replace(0, np.nan) * 100, np.nan)
    t['s_ref'] = np.where(t.vol_ref > 0, t.prop_ref / t.vol_ref.replace(0, np.nan) * 100, np.nan)
    t['delta'] = t.s_ult - t.s_ref
    s0 = t.s_ref.fillna(S_r)                      # segmento nuevo: se asume el share total anterior (efecto mezcla = 0)
    s1 = t.s_ult.fillna(s0)                       # segmento que desaparece: peso 0, no aporta
    t['mix'] = (t.w_ult - t.w_ref) * (s0 - S_r)
    t['tasa'] = t.w_ult * (s1 - s0)
    t['aporte'] = t.mix + t.tasa
    t['recuperable'] = (-t.delta).clip(lower=0).fillna(0) / 100 * t.vol_ult
    return t


def semaforo(brecha):
    """Estado según la distancia a la meta (en pp)."""
    if not _ok(brecha):
        return 'Sin dato', TINTA_SEC
    if brecha >= -UMBRAL_ALERTA:
        return 'En meta', VERDE_TEXTO
    if brecha >= -UMBRAL_CRITICO:
        return 'Alerta', AMBAR_TEXTO
    return 'Crítico', ROJO_TEXTO


@lru_cache(maxsize=64)
def contexto(ventana, comparar, deptos, oficinas, etapas, ejecs):
    """Aplica los filtros y separa los periodos a comparar. Se guarda en caché: las 4 pestañas lo comparten.

      ult  = últimos N meses de datos          prev = los N meses anteriores
      yoy  = los mismos N meses un año antes    ref  = el que eligió el usuario (prev o yoy)"""
    d = df
    if deptos:
        d = d[d.departamento.isin(deptos)]
    if oficinas:
        d = d[d.oficina.isin(oficinas)]
    if etapas:
        d = d[d.tipo_estructura.isin(etapas)]
    if ejecs:
        d = d[d.tipo_ejecucion.isin(ejecs)]
    fin = MES_FIN
    ini = mes_mas(fin, -(ventana - 1))
    fin_p = mes_mas(ini, -1)
    ini_p = mes_mas(fin_p, -(ventana - 1))
    ini_y, fin_y = mes_mas(ini, -12), mes_mas(fin, -12)

    def trozo(i, f):
        if i < MES_MIN:
            return None                                    # no hay datos tan atrás
        x = d[d.mes.between(i, f)]
        return None if x.empty else x

    ctx = SimpleNamespace(d=d, ventana=ventana, ini=ini, fin=fin, ult=d[d.mes.between(ini, fin)],
                          prev=trozo(ini_p, fin_p), yoy=trozo(ini_y, fin_y), ini_p=ini_p, fin_p=fin_p,
                          ini_y=ini_y, fin_y=fin_y)
    if comparar == 'yoy':
        ctx.ref, ctx.ini_r, ctx.fin_r, nombre = ctx.yoy, ini_y, fin_y, 'mismo periodo del año anterior'
    else:
        ctx.ref, ctx.ini_r, ctx.fin_r, nombre = ctx.prev, ini_p, fin_p, 'periodo anterior'
    ctx.ult_txt = etiqueta_periodo(ini, fin)
    ctx.ref_txt = etiqueta_periodo(ctx.ini_r, ctx.fin_r)
    ctx.ref_nombre = f'{nombre} ({ctx.ref_txt})'
    ctx.prev_txt, ctx.yoy_txt = etiqueta_periodo(ini_p, fin_p), etiqueta_periodo(ini_y, fin_y)
    return ctx


def ctx_desde(ventana, comparar, deptos, oficinas, etapas, ejecs):
    t = lambda x: tuple(sorted(x)) if x else ()
    return contexto(int(ventana or VENTANA_DEF), comparar or 'prev', t(deptos), t(oficinas), t(etapas), t(ejecs))


def maximo_reciente(ctx, meses=12):
    """Mayor valor de la media móvil de 3 meses en los últimos `meses` meses: (valor, mes). Mide cuánto se cayó desde el pico."""
    ma = serie_mensual(ctx.d)['share_ma3'].loc[mes_mas(ctx.fin, -(meses - 1)):ctx.fin].dropna()
    return (float(ma.max()), ma.idxmax()) if len(ma) else (np.nan, None)


def calcular_kpis(ctx, meta_usuario=None):
    """Todos los KPI y números clave que usan las pestañas 1, 4 y los textos de hallazgos."""
    k = SimpleNamespace()
    k.sh = share_vol(ctx.ult)
    k.sh_prev, k.sh_yoy, k.sh_ref = share_vol(ctx.prev), share_vol(ctx.yoy), share_vol(ctx.ref)
    base = k.sh_yoy if _ok(k.sh_yoy) else k.sh_prev
    k.meta_auto = base
    k.meta = float(meta_usuario) if meta_usuario not in (None, '') else base
    k.brecha = k.sh - k.meta if _ok(k.meta) else np.nan
    k.estado, k.color_estado = semaforo(k.brecha)
    k.vol_total = float(ctx.ult.volumen.sum()) if not ctx.ult.empty else 0.0
    k.rec_meta = max(0.0, -k.brecha) / 100 * k.vol_total if _ok(k.brecha) else np.nan
    k.g_propio = crecimiento(vol_propio(ctx.ult), vol_propio(ctx.ref))
    k.g_mercado = crecimiento(k.vol_total, float(ctx.ref.volumen.sum()) if ctx.ref is not None else 0)
    k.g_comp = crecimiento(vol_comp(ctx.ult), vol_comp(ctx.ref))
    k.obras, k.obras_ref = share_obras(ctx.ult), share_obras(ctx.ref)
    k.idx_precio, k.idx_precio_ref = indice_precio(ctx.ult), indice_precio(ctx.ref)
    k.p_propio, k.p_comp = precio_ponderado(ctx.ult[ctx.ult.es_propia]), precio_ponderado(ctx.ult[~ctx.ult.es_propia])
    ser = serie_mensual(ctx.d)
    k.ma3 = float(ser.share_ma3.dropna().iloc[-1]) if ser.share_ma3.notna().any() else np.nan
    k.pico, k.pico_mes = maximo_reciente(ctx)
    k.t_ofi = tabla_segmento(ctx.ult, ctx.ref, 'oficina')
    t = k.t_ofi
    k.n_ofi_datos = int(t.delta.notna().sum()) if len(t) else 0
    k.n_ofi_alerta = int(((t.delta <= -UMBRAL_OFICINA) & (t.vol_ult >= 500)).sum()) if len(t) else 0
    return k


def quiebre(ctx, k):
    """Primer mes (desde el inicio del periodo de referencia) en que la media móvil de 3 meses queda por debajo de
    la meta menos el umbral de alerta. None si no ocurre."""
    if not _ok(k.meta):
        return None
    ma = serie_mensual(ctx.d)['share_ma3'].loc[ctx.ini_r:ctx.fin].dropna()
    bajo = ma[ma < k.meta - UMBRAL_ALERTA]
    return bajo.index[0] if len(bajo) else None


def cambio_marcas(ult, ref):
    """Share de cada marca en los dos periodos y su cambio (pp)."""
    if ult is None or ref is None or ult.empty or ref.empty:
        return pd.DataFrame(columns=['s_ult', 's_ref', 'delta'])
    a = ult.groupby('marca').volumen.sum() / ult.volumen.sum() * 100
    b = ref.groupby('marca').volumen.sum() / ref.volumen.sum() * 100
    t = pd.DataFrame({'s_ult': a, 's_ref': b}).fillna(0.0)
    t['delta'] = t.s_ult - t.s_ref
    return t


def tabla_motivos(ult, ref):
    """Motivo registrado cuando la obra compró competencia (% del volumen de competencia con motivo informado)."""
    def uno(d):
        if d is None or d.empty:
            return pd.Series(dtype=float), 0.0
        c = d[~d.es_propia]
        con = c[c.motivo != 'Sin dato']
        cobertura = con.volumen.sum() / c.volumen.sum() * 100 if c.volumen.sum() > 0 else 0.0
        s = con.groupby('motivo').volumen.sum()
        return (s / s.sum() * 100 if s.sum() > 0 else s), cobertura
    (a, ca), (b, cb) = uno(ult), uno(ref)
    return pd.DataFrame({'s_ult': a, 's_ref': b}).fillna(0.0), ca, cb


def tabla_prioridad(ctx):
    """Oficinas ordenadas por volumen recuperable, con el competidor que más ganó y el motivo principal."""
    t = tabla_segmento(ctx.ult, ctx.ref, 'oficina')
    if t.empty:
        return t
    t = t[t.vol_ult > 0].copy()
    comp_top, motivo_top = {}, {}
    cu, cr = ctx.ult[~ctx.ult.es_propia], ctx.ref[~ctx.ref.es_propia]
    tot_u, tot_r = ctx.ult.groupby('oficina').volumen.sum(), ctx.ref.groupby('oficina').volumen.sum()
    mu = cu.groupby(['oficina', 'marca']).volumen.sum()
    mr = cr.groupby(['oficina', 'marca']).volumen.sum()
    for o in t.index:
        a = (mu.get(o, pd.Series(dtype=float)) / tot_u.get(o, np.nan) * 100) if o in tot_u.index else pd.Series(dtype=float)
        b = (mr.get(o, pd.Series(dtype=float)) / tot_r.get(o, np.nan) * 100) if o in tot_r.index else pd.Series(dtype=float)
        dif = a.sub(b, fill_value=0.0)
        comp_top[o] = f'{dif.idxmax()} ({dif.max():+.1f} pp)' if len(dif) and dif.max() > 0 else '—'
        m = cu[(cu.oficina == o) & (cu.motivo != 'Sin dato')].groupby('motivo').volumen.sum()
        motivo_top[o] = f'{m.idxmax()} ({m.max() / m.sum() * 100:.0f}%)' if len(m) and m.sum() > 0 else 'Sin motivo registrado'
    t['competidor'] = pd.Series(comp_top)
    t['motivo'] = pd.Series(motivo_top)
    return t.sort_values('recuperable', ascending=False)


# =====================================================================
# 3b. TEXTOS AUTOMÁTICOS (HALLAZGOS, ACCIONES, INDICADORES)
# =====================================================================
def generar_hallazgos(ctx, k):
    """Frases con lo más relevante de los datos filtrados. Describen DÓNDE y CUÁNDO; no afirman causas."""
    h = []
    if not _ok(k.sh):
        return ['No hay registros con los filtros elegidos.']
    partes = [f'El share de volumen de Cemex en {ctx.ult_txt} es {k.sh:.1f}%']
    if _ok(k.sh_prev):
        partes.append(f'{pp(k.sh - k.sh_prev)} frente al periodo anterior ({k.sh_prev:.1f}%)')
    if _ok(k.sh_yoy):
        partes.append(f'{pp(k.sh - k.sh_yoy)} frente al mismo periodo del año anterior ({k.sh_yoy:.1f}%)')
    txt = ', '.join(partes) + '.'
    if _ok(k.sh_prev) and _ok(k.sh_yoy) and k.sh - k.sh_prev < -1 and k.sh - k.sh_yoy < -1:
        txt += ' La caída también aparece contra el año anterior, así que no se explica por estacionalidad.'
    h.append(txt)

    if _ok(k.g_mercado):
        h.append(f'Contra el {ctx.ref_nombre}: el volumen censado del mercado {"crece" if k.g_mercado >= 0 else "cae"} '
                 f'{abs(k.g_mercado):.1f}%, el de Cemex {"crece" if k.g_propio >= 0 else "cae"} {abs(k.g_propio):.1f}% y el de la '
                 f'competencia {"crece" if k.g_comp >= 0 else "cae"} {abs(k.g_comp):.1f}%.'
                 + (' Cemex no acompañó el crecimiento del mercado: la competencia captura ese aumento.'
                    if k.g_mercado > 0 and k.g_propio < k.g_mercado else ''))

    t = k.t_ofi
    if len(t):
        grandes = t[(t.w_ult >= 0.02) & t.delta.notna()].sort_values('delta')
        if len(grandes):
            peores = grandes.head(3)
            h.append('Mayor caída por oficina (con al menos 2% del volumen): ' + '; '.join(
                f'{o} {r.delta:+.1f} pp ({r.s_ref:.1f}% → {r.s_ult:.1f}%)' for o, r in peores.iterrows()) + '.')
        tasa, mix, total = t.tasa.sum(), t.mix.sum(), t.aporte.sum()
        if abs(total) > 0.05:
            h.append(f'De los {pp(total)} del cambio, {pp(tasa)} ocurren dentro de cada oficina (caída real de share) y '
                     f'{pp(mix)} vienen de que cambió el peso de cada oficina en el censo (mezcla de cobertura).')
    td = tabla_segmento(ctx.ult, ctx.ref, 'departamento')
    if len(td):
        top = td.sort_values('aporte').head(2)
        if (top.aporte < -0.1).any():
            h.append('Departamentos que más restan al share total: ' + '; '.join(
                f'{o.title()} ({pp(r.aporte)})' for o, r in top.iterrows() if r.aporte < -0.1) + '.')
    for col, nombre in [('tipo_estructura', 'etapa de obra'), ('tipo_ejecucion', 'tipo de ejecución')]:
        ts = tabla_segmento(ctx.ult, ctx.ref, col)
        ts = ts[(ts.w_ult >= 0.03) & ts.delta.notna() & (ts.index != 'Sin dato')] if len(ts) else ts
        if len(ts):
            r = ts.sort_values('delta').iloc[0]
            h.append(f'Por {nombre}, la mayor caída está en "{ts.sort_values("delta").index[0]}" '
                     f'({pp(r.delta)}: {r.s_ref:.1f}% → {r.s_ult:.1f}%).')
    if _ok(k.pico) and _ok(k.ma3) and k.ma3 - k.pico < -1:
        h.append(f'Desde su máximo de los últimos 12 meses ({k.pico:.1f}% en {etiqueta_periodo(k.pico_mes, k.pico_mes)}, media móvil 3M) '
                 f'el share bajó {pp(k.ma3 - k.pico)} hasta {k.ma3:.1f}%.')
    q = quiebre(ctx, k)
    if q:
        h.append(f'La media móvil de 3 meses cae por debajo de la meta ({k.meta:.1f}%, con margen de {UMBRAL_ALERTA:.0f} pp) '
                 f'por primera vez en {etiqueta_periodo(q, q)}.')
    cm = cambio_marcas(ctx.ult, ctx.ref)
    if len(cm):
        g = cm[~cm.index.isin(MARCAS_PROPIAS)].sort_values('delta', ascending=False).head(3)
        g = g[g.delta > 0.2]
        if len(g):
            h.append('Marcas que ganan lo que Cemex pierde: ' + '; '.join(
                f'{m} {pp(r.delta)} ({r.s_ref:.1f}% → {r.s_ult:.1f}%)' for m, r in g.iterrows()) + '.')
    if _ok(k.idx_precio):
        txt = (f'Precio: Cemex vende {abs(k.idx_precio - 100):.1f}% {"por encima" if k.idx_precio > 100 else "por debajo"} de la '
               f'competencia (S/ {k.p_propio:.2f} vs S/ {k.p_comp:.2f}).')
        mot, cu, _ = tabla_motivos(ctx.ult, ctx.ref)
        if len(mot) and 'Precio' in mot.index:
            txt += f' Cuando se compra competencia y hay motivo registrado, "Precio" es el motivo en {mot.loc["Precio", "s_ult"]:.0f}% del volumen.'
        h.append(txt)
    return h


def generar_acciones(ctx, k, prio):
    """Propuesta de acciones apoyada en los números. Cada acción es una HIPÓTESIS A VALIDAR con el equipo comercial."""
    acc = []
    if len(prio):
        top = prio.head(3)
        tot_rec = prio.recuperable.sum()
        part = top.recuperable.sum() / tot_rec * 100 if tot_rec > 0 else np.nan
        acc.append(('1', 'Focalizar el plan comercial en las oficinas con mayor volumen por recuperar',
                    ', '.join(top.index),
                    f'Concentran {part:.0f}% del volumen recuperable ({formato(top.recuperable.sum())} de {formato(tot_rec)} bolsas) '
                    f'y cayeron {", ".join(f"{r.delta:+.1f} pp" for _, r in top.iterrows())} respectivamente.',
                    'Share mensual por oficina; recuperar el 50% de la brecha en 3 meses.'))
    cm = cambio_marcas(ctx.ult, ctx.ref)
    g = cm[~cm.index.isin(MARCAS_PROPIAS)].sort_values('delta', ascending=False) if len(cm) else cm
    if len(g) and g.iloc[0].delta > 0.2:
        m, r = g.index[0], g.iloc[0]
        donde = ', '.join(prio[prio.competidor.str.startswith(m + ' (')].head(3).index) if len(prio) else ''
        acc.append(('2', f'Defender frente a {m}: seguimiento de obras donde aparece y acciones de fidelización con maestros de obra y ferreterías de la zona',
                    donde or 'Zonas donde más crece',
                    f'{m} pasó de {r.s_ref:.1f}% a {r.s_ult:.1f}% ({pp(r.delta)}). '
                    + (f'Es la marca que más gana en: {donde}.' if donde else ''),
                    f'Share de {m} por oficina; volver al nivel del periodo anterior.'))
    if _ok(k.idx_precio):
        mot, _, _ = tabla_motivos(ctx.ult, ctx.ref)
        pm = mot.loc['Precio', 's_ult'] if len(mot) and 'Precio' in mot.index else np.nan
        if k.idx_precio > 100:
            texto = ('Revisar la política de precios y promociones en las oficinas donde el precio relativo es alto y el share cae '
                     '(ver gráfico de precio vs caída en la pestaña 3).')
        else:
            texto = 'El precio de Cemex no es mayor al de la competencia: priorizar servicio y disponibilidad.'
        acc.append(('3', texto, 'Oficinas con índice de precio alto y caída de share',
                    f'Índice de precio {k.idx_precio:.0f} (100 = competencia)'
                    + (f'; en el periodo anterior era {k.idx_precio_ref:.0f}' if _ok(k.idx_precio_ref) else '')
                    + (f'. "Precio" es el motivo de {pm:.0f}% del volumen de competencia con motivo informado.' if _ok(pm) else '.'),
                    'Índice de precio propio/competencia por oficina; no superar el nivel actual.'))
    peores = []
    for col, nombre in [('tipo_estructura', 'etapa'), ('tipo_ejecucion', 'ejecución')]:
        ts = tabla_segmento(ctx.ult, ctx.ref, col)
        ts = ts[(ts.w_ult >= 0.03) & ts.delta.notna() & (ts.index != 'Sin dato')] if len(ts) else ts
        if len(ts):
            i = ts.delta.idxmin()
            peores.append((nombre, i, ts.loc[i, 'delta']))
    if peores:
        acc.append(('4', 'Reforzar la presencia en los tipos de obra y etapas donde más cae el share, con ofertas para maestros y constructores '
                         '(hipótesis: la marca se define al inicio de la obra; validar con seguimiento de obras)',
                    ', '.join(f'{i} ({n})' for n, i, _ in peores),
                    '; '.join(f'En {n} "{i}" el share cambió {pp(d)}' for n, i, d in peores) + '.',
                    'Share por etapa y por tipo de ejecución.'))
    _, cob_u, _ = tabla_motivos(ctx.ult, ctx.ref)
    mix = k.t_ofi.mix.sum() if len(k.t_ofi) else np.nan
    acc.append(('5', 'Mejorar la medición: registrar el canal de venta, completar el motivo y mantener estable la cobertura del censo por oficina',
                'Todas las oficinas',
                f'No existe campo de canal (el proveedor viene vacío); el motivo solo está informado en {cob_u:.0f}% del volumen de competencia'
                + (f'; el efecto mezcla de cobertura explica {pp(mix)} del cambio.' if _ok(mix) else '.'),
                '% de registros con canal y con motivo; variación del peso de cada oficina entre periodos.'))
    return acc


def tabla_indicadores(ctx, k):
    """Tablero de indicadores de seguimiento: valor actual, umbral y estado."""
    def est(valor, malo, critico, mayor_es_mejor=True):
        if not _ok(valor):
            return 'Sin dato'
        v = valor if mayor_es_mejor else -valor
        m, c = (malo, critico) if mayor_es_mejor else (-malo, -critico)
        return 'En meta' if v >= m else ('Alerta' if v >= c else 'Crítico')
    cm = cambio_marcas(ctx.ult, ctx.ref)
    top_comp = cm[~cm.index.isin(MARCAS_PROPIAS)].s_ult.max() if len(cm) else np.nan
    _, cob_u, _ = tabla_motivos(ctx.ult, ctx.ref)
    filas = [
        ('Share de volumen (media móvil 3M)', 'Bolsas Cemex / bolsas totales censadas', 'Mensual',
         f'{k.ma3:.1f}%' if _ok(k.ma3) else '—', f'Meta {k.meta:.1f}% (alerta -{UMBRAL_ALERTA:.0f} pp, crítico -{UMBRAL_CRITICO:.0f} pp)' if _ok(k.meta) else '—',
         semaforo(k.ma3 - k.meta)[0] if _ok(k.ma3) and _ok(k.meta) else 'Sin dato'),
        ('Variación vs mismo periodo año anterior', 'Share actual - share del mismo periodo un año antes', 'Mensual',
         pp(k.sh - k.sh_yoy) if _ok(k.sh_yoy) else '—', '≥ -1 pp en meta; ≤ -3 pp crítico',
         est(k.sh - k.sh_yoy, -1, -3) if _ok(k.sh_yoy) else 'Sin dato'),
        ('Crecimiento relativo del volumen', 'Crecimiento del volumen de Cemex - crecimiento del mercado', 'Mensual',
         pp(k.g_propio - k.g_mercado) if _ok(k.g_propio) and _ok(k.g_mercado) else '—', '≥ 0 pp en meta; ≤ -5 pp crítico',
         est(k.g_propio - k.g_mercado, 0, -5) if _ok(k.g_propio) and _ok(k.g_mercado) else 'Sin dato'),
        ('Penetración en obras', '% de obras censadas donde aparece Cemex', 'Mensual',
         f'{k.obras:.1f}%' if _ok(k.obras) else '—', 'Variación ≥ -1 pp vs periodo de referencia',
         est(k.obras - k.obras_ref, -1, -3) if _ok(k.obras) and _ok(k.obras_ref) else 'Sin dato'),
        ('Índice de precio propio / competencia', 'Precio ponderado de Cemex / precio ponderado de la competencia x 100', 'Mensual',
         f'{k.idx_precio:.0f}' if _ok(k.idx_precio) else '—', '≤ 105 en meta; > 110 crítico', est(k.idx_precio, 105, 110, mayor_es_mejor=False)),
        ('Oficinas en alerta', f'Oficinas cuyo share cae ≥ {UMBRAL_OFICINA:.0f} pp', 'Mensual',
         f'{k.n_ofi_alerta} de {k.n_ofi_datos}', '0 en meta; ≥ 3 crítico',
         est(float(k.n_ofi_alerta), 0, 2, mayor_es_mejor=False) if k.n_ofi_datos else 'Sin dato'),
        ('Share de la competidora principal', 'Mayor share entre las marcas que no son Cemex', 'Mensual',
         f'{top_comp:.1f}%' if _ok(top_comp) else '—', 'Seguimiento: alerta si sube 2 pp en un semestre', 'Seguimiento'),
        ('Calidad del dato: motivo informado', '% del volumen de competencia con motivo registrado', 'Trimestral',
         f'{cob_u:.0f}%', '≥ 80% en meta', est(cob_u, 80, 50)),
    ]
    return [dict(zip(['indicador', 'definicion', 'frecuencia', 'valor', 'umbral', 'estado'], f)) for f in filas]


# =====================================================================
# 3c. FUNCIONES DE GRÁFICOS (una función = un gráfico)
# =====================================================================
# Patrón: (a) preparar los datos -> (b) crear la figura -> (c) return fig.
# Todos los dcc.Graph llevan ALTURA FIJA (ESTILO_GRAFICO): sin ella un gráfico dentro de una grilla puede crecer sin parar.
def fig_vacia(mensaje='Sin datos para los filtros elegidos'):
    fig = go.Figure()
    fig.add_annotation(text=mensaje, showarrow=False, font={'size': 15, 'color': TINTA_SEC})
    fig.update_layout(xaxis={'visible': False}, yaxis={'visible': False}, margin={'t': 30, 'b': 20},
                      plot_bgcolor='white', paper_bgcolor='white')
    return fig


def estilo(fig, titulo, leyenda=True):
    fig.update_layout(
        title={'text': titulo, 'font': {'size': 15, 'color': TINTA}, 'x': 0.01},
        plot_bgcolor='white', paper_bgcolor='white', font={'color': TINTA_SEC, 'size': 12},
        margin={'t': 85, 'l': 55, 'r': 20, 'b': 40}, showlegend=leyenda,
        legend={'orientation': 'h', 'y': 1.01, 'yanchor': 'bottom', 'x': 0, 'title': None},
        hoverlabel={'bgcolor': 'white'})
    fig.update_xaxes(showgrid=False, linecolor=BORDE)
    fig.update_yaxes(gridcolor='#e8e9f3', zeroline=False)
    return fig


def _fechas(meses):
    return pd.to_datetime([m + '-01' for m in meses])


def eje_meses(fig, cada=3):
    """Eje X con meses en español (Plotly los muestra en inglés): una marca cada `cada` meses."""
    marcas = MESES_TODOS[::cada]
    fig.update_xaxes(tickmode='array', tickvals=_fechas(marcas), ticktext=[etiqueta_periodo(m, m).replace(' ', '<br>') for m in marcas])


def graf_evolucion_share(ctx, k):
    """LÍNEAS: share mensual, media móvil de 3 meses, meta y periodo analizado sombreado."""
    s = serie_mensual(ctx.d)
    if s.share.notna().sum() == 0:
        return fig_vacia()
    x = _fechas(s.index)
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=x, y=s.share, mode='lines+markers', name='Share mensual', line={'color': AZUL_CLARO, 'width': 2},
                             marker={'size': 6}, hovertemplate='%{y:.1f}%<extra>Share mensual %{x|%m/%Y}</extra>'))
    fig.add_trace(go.Scatter(x=x, y=s.share_ma3, mode='lines', name='Media móvil 3 meses', line={'color': AZUL, 'width': 3.5},
                             hovertemplate='%{y:.1f}%<extra>Media 3M %{x|%m/%Y}</extra>'))
    if _ok(k.meta):
        fig.add_hline(y=k.meta, line_dash='dash', line_color=ROJO, line_width=2,
                      annotation_text=f'Meta {k.meta:.1f}%', annotation_position='top left', annotation_font_color=ROJO_TEXTO)
    fig.add_vrect(x0=pd.Period(ctx.ini, 'M').start_time, x1=pd.Period(ctx.fin, 'M').end_time, fillcolor=ROJO, opacity=0.07,
                  line_width=0, annotation_text=f'Periodo analizado', annotation_position='bottom right',
                  annotation_font_color=ROJO_TEXTO)
    lo = float(np.nanmin(s.share)) if s.share.notna().any() else 0
    fig.update_yaxes(ticksuffix='%', range=[max(0, np.floor(lo / 5) * 5 - 5), 100], title='Share de volumen')
    eje_meses(fig)
    return estilo(fig, 'Evolución mensual del share de volumen de Cemex')


def graf_volumen(ctx):
    """LÍNEAS: bolsas mensuales de Cemex vs competencia (¿cae Cemex o crece la competencia?)."""
    s = serie_mensual(ctx.d)
    if s.vol.notna().sum() == 0:
        return fig_vacia()
    x = _fechas(s.index)
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=x, y=s.propio, mode='lines+markers', name='Cemex', line={'color': AZUL, 'width': 3}, marker={'size': 6},
                             hovertemplate='%{y:,.0f} bolsas<extra>Cemex %{x|%m/%Y}</extra>'))
    fig.add_trace(go.Scatter(x=x, y=s.comp, mode='lines+markers', name='Competencia (todas las marcas)',
                             line={'color': COLORES_COMPETENCIA[1], 'width': 3}, marker={'size': 6},
                             hovertemplate='%{y:,.0f} bolsas<extra>Competencia %{x|%m/%Y}</extra>'))
    fig.add_vrect(x0=pd.Period(ctx.ini, 'M').start_time, x1=pd.Period(ctx.fin, 'M').end_time, fillcolor=ROJO, opacity=0.07, line_width=0)
    fig.update_yaxes(title='Bolsas censadas por mes', tickformat=',.0f', rangemode='tozero')
    eje_meses(fig)
    return estilo(fig, 'Volumen mensual censado: Cemex vs competencia')


def graf_anual(ctx):
    """LÍNEAS POR AÑO: share de cada mes del año, un trazo por año (descarta estacionalidad)."""
    s = serie_mensual(ctx.d)
    if s.share.notna().sum() == 0:
        return fig_vacia()
    s = s.assign(anio=[m[:4] for m in s.index], mes_n=[int(m[5:]) for m in s.index])
    anios = sorted(s.anio.unique())
    rampa = px.colors.sample_colorscale([[0, AZUL_CLARO], [1, AZUL]], [i / max(len(anios) - 1, 1) for i in range(len(anios))])
    fig = go.Figure()
    for a, col in zip(anios, rampa):
        z = s[s.anio == a]
        ultimo = a == anios[-1]
        fig.add_trace(go.Scatter(x=[MESES_ES[i - 1].capitalize() for i in z.mes_n], y=z.share, mode='lines+markers', name=a,
                                 line={'color': col, 'width': 4 if ultimo else 2}, marker={'size': 7 if ultimo else 5},
                                 hovertemplate='%{y:.1f}%<extra>' + a + ' %{x}</extra>'))
    fig.update_yaxes(ticksuffix='%', title='Share de volumen')
    fig.update_xaxes(categoryorder='array', categoryarray=[m.capitalize() for m in MESES_ES])
    return estilo(fig, 'Share por mes del año: comparación entre años')


def graf_puente(t, ctx):
    """CASCADA (desde 0): cómo se compone el cambio total del share = efecto mezcla + efecto tasa por oficina."""
    if t.empty:
        return fig_vacia()
    total = t.aporte.sum()
    tasa = t.tasa.sort_values()
    n = 5
    peores = tasa[tasa < 0].head(n)
    resto = t.tasa.sum() - peores.sum()
    x = ['Efecto mezcla<br>(cobertura)'] + [f'{o}' for o in peores.index] + ['Otras oficinas', 'Cambio total']
    y = [t.mix.sum()] + peores.tolist() + [resto, 0]
    fig = go.Figure(go.Waterfall(
        x=x, y=y, measure=['relative'] * (len(x) - 1) + ['total'],
        text=[f'{v:+.2f}' for v in y[:-1]] + [f'{total:+.2f}'], textposition='outside',
        decreasing={'marker': {'color': ROJO}}, increasing={'marker': {'color': AZUL_MEDIO}}, totals={'marker': {'color': AZUL}},
        connector={'line': {'color': COLOR_OTRAS, 'width': 1}},
        hovertemplate='%{x}: %{y:+.2f} pp<extra></extra>'))
    fig.update_yaxes(title='Puntos porcentuales (pp)', zeroline=True, zerolinecolor=TINTA_SUAVE)
    fig.update_xaxes(tickangle=0)
    estilo(fig, f'¿De dónde viene el cambio? ({ctx.ref_txt} → {ctx.ult_txt})', leyenda=False)
    fig.update_layout(margin={'t': 85, 'l': 55, 'r': 20, 'b': 70})
    return fig


def graf_delta_segmento(t, nombre):
    """BARRAS HORIZONTALES: cambio de share (pp) por segmento. Rojo = cae, azul = sube."""
    t = t[t.delta.notna() & (t.vol_ult > 0) & (t.vol_ref > 0)]
    if t.empty:
        return fig_vacia()
    t = t.sort_values('vol_ult', ascending=False).head(15).sort_values('delta', ascending=False)
    fig = go.Figure(go.Bar(
        x=t.delta, y=t.index.astype(str), orientation='h', marker_color=[ROJO if v < 0 else AZUL for v in t.delta],
        text=[f'{v:+.1f} pp' for v in t.delta], textposition='outside', cliponaxis=False,
        customdata=np.stack([t.s_ref, t.s_ult, t.vol_ult], axis=1),
        hovertemplate='<b>%{y}</b><br>Share: %{customdata[0]:.1f}% → %{customdata[1]:.1f}%<br>Cambio: %{x:+.1f} pp'
                      '<br>Volumen actual: %{customdata[2]:,.0f} bolsas<extra></extra>'))
    lim = float(np.nanmax(np.abs(t.delta))) * 1.25 + 1
    fig.update_xaxes(range=[-lim, lim], ticksuffix=' pp', zeroline=True, zerolinecolor=TINTA_SUAVE)
    fig.update_yaxes(showgrid=False)
    estilo(fig, f'Cambio del share por {nombre.lower()} (pp)', leyenda=False)
    fig.update_layout(margin={'t': 85, 'l': 150, 'r': 50, 'b': 40})
    return fig


def graf_aporte_segmento(t, nombre):
    """BARRAS APILADAS: cuánto resta o suma cada segmento al share total (efecto tasa + efecto mezcla)."""
    t = t[t.aporte.notna() & (t.vol_ult > 0)]
    if t.empty:
        return fig_vacia()
    t = t.reindex(t.aporte.abs().sort_values(ascending=False).head(15).index).sort_values('aporte', ascending=False)
    y = [f'{i}  ({v:+.2f})' for i, v in zip(t.index.astype(str), t.aporte)]          # el aporte total va en la etiqueta
    fig = go.Figure()
    fig.add_trace(go.Bar(x=t.tasa, y=y, orientation='h', name='Efecto tasa (cambió su share)', marker_color=AZUL,
                         hovertemplate='%{y}<br>Efecto tasa: %{x:+.2f} pp<extra></extra>'))
    fig.add_trace(go.Bar(x=t.mix, y=y, orientation='h', name='Efecto mezcla (cambió su peso)', marker_color=COLOR_OTRAS,
                         hovertemplate='%{y}<br>Efecto mezcla: %{x:+.2f} pp<extra></extra>'))
    fig.add_trace(go.Scatter(x=t.aporte, y=y, mode='markers', name='Aporte total (entre paréntesis)',
                             marker={'symbol': 'diamond', 'size': 9, 'color': ROJO, 'line': {'color': 'white', 'width': 1.5}},
                             hovertemplate='%{y}<br>Aporte total: %{x:+.2f} pp<extra></extra>'))
    fig.update_layout(barmode='relative')
    fig.update_xaxes(ticksuffix=' pp', zeroline=True, zerolinecolor=TINTA_SUAVE)
    fig.update_yaxes(showgrid=False, categoryorder='array', categoryarray=y)
    estilo(fig, f'Aporte de cada {nombre.lower()} al cambio del share total (pp)')
    fig.update_layout(margin={'t': 110, 'l': 190, 'r': 30, 'b': 40}, legend={'y': 1.0, 'yanchor': 'bottom', 'x': 0, 'orientation': 'h'})
    return fig


_COORD = {}


def _contorno(ubigeo):
    """Contorno del distrito como listas x (lon) / y (lat) separadas por None, listas para go.Scatter."""
    if ubigeo not in _COORD:
        geom = GEO[ubigeo]['geometry']
        poligonos = [geom['coordinates']] if geom['type'] == 'Polygon' else geom['coordinates']
        xs, ys = [], []
        for poligono in poligonos:
            for anillo in poligono:
                a = np.asarray(anillo)[:, :2]
                xs += a[:, 0].tolist() + [None]
                ys += a[:, 1].tolist() + [None]
        _COORD[ubigeo] = (xs, ys)
    return _COORD[ubigeo]


_FONDO = {}


def _fondo_peru():
    """Contorno de TODOS los distritos del geojson en una sola línea (se calcula una vez)."""
    if 'xy' not in _FONDO:
        xs, ys = [], []
        for ub in GEO:
            x, y = _contorno(ub)
            xs += x
            ys += y
        _FONDO['xy'] = (xs, ys)
    return _FONDO['xy']


def tabla_distritos(ctx):
    """Tabla por distrito (ubigeo) con nombres y la oficina que más lo visita."""
    t = tabla_segmento(ctx.ult, ctx.ref, 'ubigeo')
    if t.empty:
        return t
    t = t.reset_index().merge(NOMBRES_UBIGEO_DF, on='ubigeo', how='left')
    ofi = ctx.ult.groupby('ubigeo').oficina.agg(lambda s: s.mode().iat[0])
    t['oficina'] = t.ubigeo.map(ofi)
    for c in ['distrito', 'provincia', 'departamento', 'oficina']:
        t[c] = t[c].fillna('Sin dato')
    return t


def graf_mapa_delta(t, minimo, vol_total):
    """MAPA (polígonos, sin internet): cambio del share por distrito. Rojo = cae, azul = sube.
    Los distritos sin polígono en el geojson se dibujan como burbujas en el centro de sus visitas."""
    if not GEO:
        return fig_vacia('Falta el archivo peru_distrital_simple.geojson')
    r = t[(t.vol_ult >= minimo) & (t.vol_ref >= minimo)].dropna(subset=['delta']) if len(t) else t
    if r.empty:
        return fig_vacia('Ningún distrito alcanza el volumen mínimo elegido en ambos periodos')
    lim = float(np.clip(r.delta.abs().quantile(0.95), 5, 25))
    en_geo = r[r.ubigeo.isin(GEO)]
    sin_geo = r[~r.ubigeo.isin(GEO)].join(CENTROIDES, on='ubigeo').dropna(subset=['lat'])
    texto = lambda x: (x.distrito + ' (' + x.provincia + ', ' + x.departamento + ')<br>Share: ' + x.s_ref.round(1).astype(str) + '% → ' +
                       x.s_ult.round(1).astype(str) + '%<br>Cambio: ' + x.delta.round(1).map('{:+.1f}'.format) + ' pp<br>Volumen actual: ' +
                       x.vol_ult.round(0).map('{:,.0f}'.format) + ' bolsas')
    fig = go.Figure()
    fx, fy = _fondo_peru()                           # mapa completo del Perú: distritos sin dato quedan en blanco, solo con contorno
    fig.add_trace(go.Scatter(x=fx, y=fy, mode='lines', fill='toself', fillcolor='#fbfbfe', hoverinfo='skip', showlegend=False,
                             line={'color': '#c3c6dc', 'width': 0.5}))
    colores = px.colors.sample_colorscale(ESCALA_DELTA, list(np.clip((en_geo.delta + lim) / (2 * lim), 0, 1)))
    for (_, fila), txt, col in zip(en_geo.iterrows(), texto(en_geo), colores):
        xs, ys = _contorno(fila.ubigeo)
        fig.add_trace(go.Scatter(x=xs, y=ys, mode='lines', fill='toself', fillcolor=col, hoveron='fills',
                                 line={'color': '#8d90ad', 'width': 0.4}, text=txt, hoverinfo='text', showlegend=False))
    if len(sin_geo):
        fig.add_trace(go.Scatter(
            x=sin_geo.lon, y=sin_geo.lat, mode='markers', text=texto(sin_geo) + '<br>(sin polígono: punto en el centro de sus visitas)',
            hoverinfo='text', showlegend=False,
            marker={'size': np.clip(sin_geo.vol_ult / max(vol_total, 1) * 400, 10, 34), 'color': sin_geo.delta,
                    'colorscale': ESCALA_DELTA, 'cmin': -lim, 'cmax': lim, 'line': {'color': 'white', 'width': 1.5}}))
    fig.add_trace(go.Scatter(x=[None], y=[None], mode='markers', hoverinfo='skip', showlegend=False,
                             marker={'colorscale': ESCALA_DELTA, 'cmin': -lim, 'cmax': lim, 'color': [0], 'showscale': True,
                                     'colorbar': {'title': 'Δ share (pp)', 'thickness': 12, 'len': 0.7}}))
    # Vista inicial: acercada a la zona con datos (el resto del Perú sigue dibujado: se llega con zoom/arrastre).
    xs_v, ys_v = [], []
    for ub in en_geo.ubigeo:
        x, y = _contorno(ub)
        xs_v += [v for v in x if v is not None]
        ys_v += [v for v in y if v is not None]
    xs_v += sin_geo.lon.tolist()
    ys_v += sin_geo.lat.tolist()
    rango_x = rango_y = None
    if xs_v:
        mx, my = (max(xs_v) - min(xs_v)) * 0.08 + 0.1, (max(ys_v) - min(ys_v)) * 0.08 + 0.1
        rango_x, rango_y = [min(xs_v) - mx, max(xs_v) + mx], [min(ys_v) - my, max(ys_v) + my]
    fig.add_annotation(text='Rueda del mouse = zoom · arrastrar = mover · doble clic = vista inicial', xref='paper', yref='paper', x=0.01, y=0.0,
                       showarrow=False, xanchor='left', yanchor='bottom', font={'size': 11, 'color': TINTA_SUAVE})
    fig.update_xaxes(visible=False, range=rango_x)
    fig.update_yaxes(visible=False, scaleanchor='x', scaleratio=1, range=rango_y)
    fig.update_layout(dragmode='pan', margin={'t': 55, 'l': 0, 'r': 0, 'b': 0}, paper_bgcolor='white', plot_bgcolor='white',
                      title={'text': 'Cambio del share por distrito (rojo = cae, azul = sube, blanco = sin datos suficientes)', 'font': {'size': 15, 'color': TINTA}, 'x': 0.01})
    return fig


def graf_heatmap(ctx):
    """MAPA DE CALOR: share por oficina y mes. Muestra DÓNDE y CUÁNDO se concentra la caída."""
    meses = MESES_TODOS[-MESES_HEATMAP:]
    d = ctx.d[ctx.d.mes.isin(meses)]
    if d.empty:
        return fig_vacia()
    g = d.assign(propio=np.where(d.es_propia, d.volumen, 0.0)).groupby(['oficina', 'mes']).agg(v=('volumen', 'sum'), p=('propio', 'sum'))
    g['share'] = g.p / g.v * 100
    z = g.share.unstack('mes').reindex(columns=meses)
    z = z.loc[z.iloc[:, -ctx.ventana:].mean(axis=1).sort_values().index]            # oficinas con menor share reciente arriba
    fig = go.Figure(go.Heatmap(
        z=z.values, x=[etiqueta_periodo(m, m) for m in z.columns], y=z.index, colorscale=ESCALA_DELTA,
        zmin=float(np.floor(np.nanpercentile(z.values, 5) / 10) * 10), zmax=100, xgap=1, ygap=1,
        texttemplate='%{z:.0f}', textfont={'size': 10}, hoverongaps=False,
        colorbar={'title': 'Share %', 'thickness': 12},
        hovertemplate='<b>%{y}</b> · %{x}<br>Share: %{z:.1f}%<extra></extra>'))
    fig.update_yaxes(autorange='reversed', showgrid=False)
    fig.update_xaxes(showgrid=False, side='bottom', tickangle=-45)
    estilo(fig, f'Share (%) por oficina y mes · últimos {len(meses)} meses', leyenda=False)
    fig.update_layout(margin={'t': 55, 'l': 110, 'r': 20, 'b': 70})
    return fig


def graf_cambio_marcas(cm):
    """BARRAS HORIZONTALES: cambio de share (pp) de cada marca. ¿Quién captura lo que se pierde?"""
    if cm.empty:
        return fig_vacia()
    cm = cm[(cm.s_ult >= 0.3) | (cm.s_ref >= 0.3)].sort_values('delta', ascending=False)
    fig = go.Figure(go.Bar(
        x=cm.delta, y=cm.index, orientation='h', marker_color=[color_marca(m) for m in cm.index],
        text=[f'{v:+.1f} pp' for v in cm.delta], textposition='outside', cliponaxis=False,
        customdata=np.stack([cm.s_ref, cm.s_ult], axis=1),
        hovertemplate='<b>%{y}</b><br>Share: %{customdata[0]:.1f}% → %{customdata[1]:.1f}%<br>Cambio: %{x:+.1f} pp<extra></extra>'))
    lim = float(cm.delta.abs().max()) * 1.3 + 0.5
    fig.update_xaxes(range=[-lim, lim], ticksuffix=' pp', zeroline=True, zerolinecolor=TINTA_SUAVE)
    fig.update_yaxes(showgrid=False)
    estilo(fig, '¿Quién captura el share? Cambio por marca (pp)', leyenda=False)
    fig.update_layout(margin={'t': 85, 'l': 110, 'r': 50, 'b': 40})
    return fig


def graf_competidoras(ctx):
    """LÍNEAS: evolución del share de las marcas competidoras principales."""
    d = ctx.d[ctx.d.marca.isin(TOP_COMPETENCIA)]
    if d.empty:
        return fig_vacia()
    tot = ctx.d.groupby('mes').volumen.sum()
    v = d.groupby(['marca', 'mes']).volumen.sum().unstack('marca').reindex(MESES_TODOS)
    s = v.div(tot.reindex(MESES_TODOS), axis=0) * 100
    x = _fechas(MESES_TODOS)
    fig = go.Figure()
    for m in TOP_COMPETENCIA:
        if m in s.columns:
            fig.add_trace(go.Scatter(x=x, y=s[m], mode='lines+markers', name=m, line={'color': color_marca(m), 'width': 3}, marker={'size': 5},
                                     hovertemplate='%{y:.1f}%<extra>' + m + ' %{x|%m/%Y}</extra>'))
    fig.add_vrect(x0=pd.Period(ctx.ini, 'M').start_time, x1=pd.Period(ctx.fin, 'M').end_time, fillcolor=ROJO, opacity=0.07, line_width=0)
    fig.update_yaxes(ticksuffix='%', title='Share de volumen', rangemode='tozero')
    eje_meses(fig)
    return estilo(fig, 'Evolución del share de las principales competidoras')


def graf_precio(ctx):
    """LÍNEAS: precio promedio ponderado por volumen, Cemex vs competencia."""
    d = ctx.d
    x = d[(d.precio > 0) & (d.volumen > 0)]
    if len(x) < 10:
        return fig_vacia('Pocos precios registrados')
    lo, hi = x.precio.quantile([0.01, 0.99])
    x = x[x.precio.between(lo, hi)].assign(pv=lambda t: t.precio * t.volumen, grupo=lambda t: np.where(t.es_propia, 'Cemex', 'Competencia'))
    r = x.groupby(['mes', 'grupo']).agg(pv=('pv', 'sum'), v=('volumen', 'sum')).reset_index()
    r['precio'] = r.pv / r.v
    fig = go.Figure()
    for g, col in [('Cemex', AZUL), ('Competencia', COLOR_OTRAS)]:
        z = r[r.grupo == g].set_index('mes').reindex(MESES_TODOS)
        fig.add_trace(go.Scatter(x=_fechas(MESES_TODOS), y=z.precio, mode='lines+markers', name=g, line={'color': col, 'width': 3},
                                 marker={'size': 5}, hovertemplate='S/ %{y:.2f}<extra>' + g + ' %{x|%m/%Y}</extra>'))
    fig.update_yaxes(title='Precio ponderado (S/ por bolsa)', tickprefix='S/ ')
    eje_meses(fig)
    return estilo(fig, 'Precio promedio ponderado por volumen: Cemex vs competencia')


def graf_precio_vs_caida(ctx):
    """DISPERSIÓN: ¿las oficinas con precio relativo más alto son las que más caen?"""
    t = tabla_segmento(ctx.ult, ctx.ref, 'oficina')
    if t.empty:
        return fig_vacia()
    filas = []
    for o in t.index:
        u = ctx.ult[ctx.ult.oficina == o]
        if len(u) < 30 or not _ok(t.loc[o, 'delta']) or t.loc[o, 'vol_ref'] <= 0:
            continue
        i = indice_precio(u)
        if _ok(i):
            filas.append((o, i, t.loc[o, 'delta'], t.loc[o, 'vol_ult']))
    r = pd.DataFrame(filas, columns=['oficina', 'idx', 'delta', 'vol'])
    if len(r) < 4:
        return fig_vacia('Pocas oficinas con precios para comparar')
    corr = float(np.corrcoef(r.idx, r.delta)[0, 1])
    fig = go.Figure(go.Scatter(
        x=r.idx, y=r.delta, mode='markers+text', text=r.oficina, textposition='top center', textfont={'size': 10},
        marker={'size': np.clip(r.vol / r.vol.max() * 40, 9, 40), 'color': [ROJO if v < 0 else AZUL for v in r.delta],
                'opacity': 0.8, 'line': {'color': 'white', 'width': 1.5}},
        hovertemplate='<b>%{text}</b><br>Índice de precio: %{x:.0f}<br>Cambio de share: %{y:+.1f} pp<extra></extra>'))
    if len(r) >= 6:
        b, a = np.polyfit(r.idx, r.delta, 1)
        xs = np.array([r.idx.min(), r.idx.max()])
        fig.add_trace(go.Scatter(x=xs, y=a + b * xs, mode='lines', line={'color': TINTA_SUAVE, 'dash': 'dot', 'width': 2}, hoverinfo='skip'))
    fig.add_vline(x=100, line_color=BORDE)
    fig.add_hline(y=0, line_color=BORDE)
    fig.add_annotation(text=f'Correlación r = {corr:+.2f} (n = {len(r)} oficinas). Correlación no es causalidad.',
                       xref='paper', yref='paper', x=0.01, y=-0.3, showarrow=False, font={'size': 11, 'color': TINTA_SUAVE}, xanchor='left')
    fig.update_xaxes(title='Índice de precio Cemex / competencia (100 = igual)')
    fig.update_yaxes(title='Cambio de share (pp)', ticksuffix=' pp')
    estilo(fig, '¿El precio explica la caída? Precio relativo vs cambio de share', leyenda=False)
    fig.update_layout(margin={'t': 70, 'l': 60, 'r': 20, 'b': 100})
    return fig


def graf_motivos(ult, ref, ctx):
    """BARRAS AGRUPADAS: por qué la obra compró competencia (solo donde hay motivo registrado)."""
    m, cob_u, cob_r = tabla_motivos(ult, ref)
    if m.empty:
        return fig_vacia('Sin motivos registrados')
    m = m.assign(mx=m[['s_ult', 's_ref']].max(axis=1)).sort_values('mx', ascending=False).head(7).sort_values('mx')
    fig = go.Figure()
    fig.add_trace(go.Bar(y=m.index, x=m.s_ref, orientation='h', name=f'Referencia ({ctx.ref_txt})', marker_color=AZUL_CLARO,
                         hovertemplate='%{x:.1f}%<extra>Referencia</extra>'))
    fig.add_trace(go.Bar(y=m.index, x=m.s_ult, orientation='h', name=f'Actual ({ctx.ult_txt})', marker_color=AZUL,
                         text=[f'{v:.0f}%' for v in m.s_ult], textposition='outside', cliponaxis=False,
                         hovertemplate='%{x:.1f}%<extra>Actual</extra>'))
    fig.update_layout(barmode='group')
    fig.update_xaxes(ticksuffix='%', title='% del volumen de competencia con motivo informado')
    fig.update_yaxes(showgrid=False)
    estilo(fig, 'Motivo de compra de la competencia', leyenda=True)
    fig.add_annotation(text=f'Motivo informado solo en {cob_u:.0f}% del volumen de competencia actual.', xref='paper', yref='paper',
                       x=0.0, y=-0.3, showarrow=False, font={'size': 11, 'color': TINTA_SUAVE}, xanchor='left')
    fig.update_layout(margin={'t': 85, 'l': 150, 'r': 40, 'b': 105})
    return fig


def graf_simulador(prio, pct, n, k):
    """CASCADA (desde 0): cuántos pp de share se ganarían recuperando parte de la brecha en las oficinas prioritarias."""
    if prio.empty or k.vol_total <= 0:
        return fig_vacia()
    top = prio.head(n)
    gan = pct / 100 * top.recuperable / k.vol_total * 100
    total = gan.sum()
    x = list(top.index) + ['Share ganado']
    y = gan.tolist() + [0]
    fig = go.Figure(go.Waterfall(x=x, y=y, measure=['relative'] * len(gan) + ['total'],
                                 text=[f'+{v:.2f}' for v in gan] + [f'+{total:.2f}'], textposition='outside',
                                 increasing={'marker': {'color': AZUL_MEDIO}}, totals={'marker': {'color': AZUL}},
                                 connector={'line': {'color': COLOR_OTRAS, 'width': 1}},
                                 hovertemplate='%{x}: %{y:+.2f} pp<extra></extra>'))
    if _ok(k.brecha) and k.brecha < 0:
        fig.add_hline(y=-k.brecha, line_dash='dash', line_color=ROJO, line_width=2,
                      annotation_text=f'Brecha a la meta: {-k.brecha:.1f} pp', annotation_position='top left',
                      annotation_font_color=ROJO_TEXTO)
    fig.update_yaxes(title='Puntos porcentuales de share ganados', rangemode='tozero')
    estilo(fig, f'Simulador: {pct}% de la caída recuperada en {len(top)} oficina(s) prioritaria(s)', leyenda=False)
    fig.update_layout(margin={'t': 85, 'l': 55, 'r': 20, 'b': 70})
    return fig


# =====================================================================
# 3d. MOTOR DE CONSULTAS SQL (pestaña 5)
# =====================================================================
# Capas de seguridad: (1) usuario de solo lectura, (2) solo UNA sentencia SELECT/WITH, (3) lista de palabras
# prohibidas, (4) solo tablas permitidas, (5) máximo de filas y de tiempo.
TABLAS_DATAMART = ['vw_consumo', 'fact_consumo', 'Tiempo', 'Oficina', 'Ubigeo', 'Obra', 'TipoEstructura', 'Marca',
                   'UnidadMedida', 'Motivo', 'Ejecutivo', 'Proveedor', 'Competencia']
TABLAS_PERMITIDAS = {t.lower() for t in TABLAS_DATAMART}

PALABRAS_PROHIBIDAS = re.compile(
    r'\b(INSERT|UPDATE|DELETE|DROP|ALTER|CREATE|TRUNCATE|MERGE|EXEC|EXECUTE|GRANT|REVOKE|DENY|BACKUP|RESTORE|'
    r'SHUTDOWN|OPENROWSET|OPENQUERY|OPENDATASOURCE|BULK|INTO|WAITFOR|DBCC|KILL|RECONFIGURE|USE|ATTACH|DETACH|PRAGMA)\b'
    r'|\b(xp_|sp_)\w*', re.IGNORECASE)

DICCIONARIO_VISTA = [
    ('id_encuesta', 'INT', 'Código de la encuesta (1 fila = 1 encuesta)'),
    ('id_visita', 'INT', 'Código de la visita'),
    ('id_obra', 'BIGINT', 'Código de la obra'),
    ('fecha', 'DATE', 'Fecha de la encuesta'),
    ('anio', 'SMALLINT', 'Año'),
    ('mes', 'TEXTO', 'Año-mes, por ejemplo 2025-03'),
    ('oficina', 'TEXTO', 'Oficina comercial'),
    ('ubigeo', 'CHAR(6)', 'Código INEI del distrito'),
    ('departamento', 'TEXTO', ''), ('provincia', 'TEXTO', ''), ('distrito', 'TEXTO', ''),
    ('tipo_estructura', 'TEXTO', 'Etapa / estructura de la obra'),
    ('tipo_ejecucion', 'TEXTO', 'Cómo se ejecuta la obra'),
    ('marca', 'TEXTO', 'Marca (nombre más frecuente)'),
    ('marca_norm', 'TEXTO', 'Marca en mayúsculas; une "Holcim" y "HOLCIM". Úsela para agrupar'),
    ('es_propia', 'BIT', '1 = marca propia (Cemex Cemento)'),
    ('producto', 'TEXTO', 'Cemento, Concreto, etc.'),
    ('linea', 'TEXTO', ''), ('sublinea', 'TEXTO', ''),
    ('unidad', 'TEXTO', 'Unidad de medida (BLS, TM...). No sume unidades distintas'),
    ('motivo', 'TEXTO', 'Motivo de la elección de marca'),
    ('ejecutivo', 'TEXTO', 'Ejecutivo que creó la encuesta'),
    ('proveedor_codigo', 'BIGINT', 'Código del proveedor'),
    ('tipo_competencia', 'TEXTO', ''), ('marca_competencia', 'TEXTO', 'Marca de cemento de la competencia'),
    ('volumen', 'DECIMAL', 'Volumen censado, en la unidad indicada'),
    ('volumen_restante', 'DECIMAL', 'Volumen restante censado'),
    ('precio', 'DECIMAL', 'Precio censado'),
    ('tipo_visita', 'TINYINT', ''),
    ('estado', 'BIT', '1 = activa, 0 = dada de baja'),
    ('lat', 'DECIMAL', 'Latitud del check-in'), ('lon', 'DECIMAL', 'Longitud del check-in'),
]

ESQUEMA_ESTRELLA = {
    'fact_consumo': 'id, visitaID, tiempoID, oficinaID, ubigeoID, obraID, tipoEstructuraID, marcaID, unidadMedidaID, '
                    'motivoID, ejecutivoID, proveedorID, competenciaID, idSociedad, sociedadCli6052, sociedadCli6012, '
                    'tipoVisita, estado, posibilidadReversion, flagMotivoCpsaa, volumen, volumenRestante, precio, '
                    'fechaIngreso, fechaCreacion, fechaModificacion, checkin, checkout, duracionMin, checkinLatitud, '
                    'checkinLongitud, checkoutLatitud, checkoutLongitud, acciones, observacion',
    'Tiempo': 'id, fecha, anio, trimestre, numMes, mes, anioMes, dia, semanaISO, diaSemana, nombreDia, esFinDeSemana',
    'Oficina': 'id, codigo, nombre, idSociedad',
    'Ubigeo': 'id, codigo, codDepartamento, codProvincia, codDistrito, departamento, provincia, distrito',
    'Obra': 'id, codigo, idMaestroObra, tipoEjecucion',
    'TipoEstructura': 'id, codigo, nombre',
    'Marca': 'id, codigo, tipoMarca, codMarca, marca, marcaNormalizada, esMarcaPropia, codLinea, linea, codSublinea, '
             'sublinea, codProducto, producto',
    'UnidadMedida': 'id, codigo, abreviatura', 'Motivo': 'id, codigo, nombre',
    'Ejecutivo': 'id, codigo, ejecutivoSAP, nombre', 'Proveedor': 'id, codigo, nombre',
    'Competencia': 'id, codTipoCompetencia, tipoCompetencia, codMarcaCemento, marcaCemento',
}

_BASE = "FROM vw_consumo\nWHERE producto = 'Cemento' AND unidad = 'BLS' AND estado = 1"
_SEM = ("CASE WHEN fecha >= '2025-07-01' THEN 'Jul-Dic 2025' ELSE 'Ene-Jun 2025' END")
EJEMPLOS = {
    'Share de Cemex por mes': f"""SELECT mes,
       SUM(CASE WHEN es_propia = 1 THEN volumen ELSE 0 END) AS bolsas_cemex,
       SUM(volumen) AS bolsas_total,
       ROUND(100.0 * SUM(CASE WHEN es_propia = 1 THEN volumen ELSE 0 END) / NULLIF(SUM(volumen), 0), 1) AS share_pct
{_BASE}
GROUP BY mes
ORDER BY mes""",
    'Caída del share por oficina (Ene-Jun vs Jul-Dic 2025)': f"""WITH s AS (
    SELECT oficina,
           ROUND(100.0 * SUM(CASE WHEN fecha < '2025-07-01' AND es_propia = 1 THEN volumen ELSE 0 END)
                 / NULLIF(SUM(CASE WHEN fecha < '2025-07-01' THEN volumen ELSE 0 END), 0), 1) AS share_ene_jun,
           ROUND(100.0 * SUM(CASE WHEN fecha >= '2025-07-01' AND es_propia = 1 THEN volumen ELSE 0 END)
                 / NULLIF(SUM(CASE WHEN fecha >= '2025-07-01' THEN volumen ELSE 0 END), 0), 1) AS share_jul_dic,
           SUM(CASE WHEN fecha >= '2025-07-01' THEN volumen ELSE 0 END) AS bolsas_jul_dic
    {_BASE} AND fecha >= '2025-01-01'
    GROUP BY oficina
)
SELECT oficina, share_ene_jun, share_jul_dic,
       ROUND(share_jul_dic - share_ene_jun, 1) AS variacion_pp, bolsas_jul_dic
FROM s
ORDER BY variacion_pp""",
    'Share por marca y semestre (¿quién gana?)': f"""SELECT {_SEM} AS periodo, marca_norm AS marca, SUM(volumen) AS bolsas,
       ROUND(100.0 * SUM(volumen) / SUM(SUM(volumen)) OVER (PARTITION BY {_SEM}), 1) AS share_pct
{_BASE} AND fecha >= '2025-01-01'
GROUP BY {_SEM}, marca_norm
ORDER BY periodo, bolsas DESC""",
    'Motivos de compra de la competencia (Jul-Dic 2025)': f"""SELECT motivo, SUM(volumen) AS bolsas,
       ROUND(100.0 * SUM(volumen) / SUM(SUM(volumen)) OVER (), 1) AS pct
{_BASE} AND es_propia = 0 AND fecha >= '2025-07-01'
GROUP BY motivo
ORDER BY bolsas DESC""",
    'Precio ponderado por mes: Cemex vs competencia': f"""SELECT mes,
       ROUND(SUM(CASE WHEN es_propia = 1 THEN precio * volumen END) / NULLIF(SUM(CASE WHEN es_propia = 1 THEN volumen END), 0), 2) AS precio_cemex,
       ROUND(SUM(CASE WHEN es_propia = 0 THEN precio * volumen END) / NULLIF(SUM(CASE WHEN es_propia = 0 THEN volumen END), 0), 2) AS precio_competencia
{_BASE} AND precio > 0 AND volumen > 0
GROUP BY mes
ORDER BY mes""",
    'Top 10 distritos con más volumen no capturado (2025)': f"""SELECT TOP 10 departamento, provincia, distrito,
       SUM(volumen) AS bolsas_mercado,
       SUM(CASE WHEN es_propia = 1 THEN volumen ELSE 0 END) AS bolsas_cemex,
       SUM(CASE WHEN es_propia = 0 THEN volumen ELSE 0 END) AS bolsas_competencia
{_BASE} AND fecha >= '2025-01-01'
GROUP BY departamento, provincia, distrito
ORDER BY bolsas_competencia DESC""",
    'Últimas 100 encuestas (todas las columnas)': "SELECT TOP 100 *\nFROM vw_consumo\nORDER BY fecha DESC",
    'JOIN sobre el modelo estrella (fact_consumo + dimensiones)': """SELECT TOP 20 o.nombre AS oficina, m.marca, SUM(f.volumen) AS volumen
FROM fact_consumo f
JOIN Oficina o ON f.oficinaID = o.id
JOIN Marca   m ON f.marcaID   = m.id
GROUP BY o.nombre, m.marca
ORDER BY volumen DESC""",
}

_TEXTO_O_COMENTARIO = re.compile(r"'(?:[^']|'')*'|/\*.*?\*/|--[^\n]*", re.S)


def sin_comentarios(sql):
    """Quita los comentarios (-- y /* */) respetando los textos entre comillas simples."""
    return _TEXTO_O_COMENTARIO.sub(lambda m: m.group(0) if m.group(0).startswith("'") else ' ', sql)


def _sin_literales_ni_comentarios(sql):
    return re.sub(r"'(?:[^']|'')*'", "''", sin_comentarios(sql))      # sin comentarios y con los textos vaciados


def validar_consulta(sql):
    """Devuelve (ok, mensaje). Acepta solo UNA sentencia SELECT / WITH sobre las tablas permitidas."""
    if not sql or not sql.strip():
        return False, 'Escriba una consulta.'
    limpio = _sin_literales_ni_comentarios(sql).strip().rstrip(';').strip()
    if not limpio:
        return False, 'La consulta está vacía (solo comentarios).'
    if ';' in limpio:
        return False, 'Solo se permite una consulta a la vez (quite los ";" intermedios).'
    if not re.match(r'(SELECT|WITH)\b', limpio, re.IGNORECASE):
        return False, 'Solo se permiten consultas que empiecen con SELECT (o WITH).'
    m = PALABRAS_PROHIBIDAS.search(limpio)
    if m:
        return False, f'No permitido: "{m.group(0)}". Esta consola es solo de lectura.'
    if sqlglot is not None:
        try:
            arbol = sqlglot.parse_one(sin_comentarios(sql).strip().rstrip(';').strip(), read='tsql')
        except Exception as e:                                # error de sintaxis
            return False, f'Error de sintaxis: {str(e).splitlines()[0]}'
        if not isinstance(arbol, (exp.Select, exp.Union, exp.Subquery)):
            return False, 'Solo se permiten consultas SELECT.'
        ctes = {c.alias_or_name.lower() for c in arbol.find_all(exp.CTE)}
        for t in arbol.find_all(exp.Table):
            nombre = t.name.lower()
            if nombre in ctes and not t.db:
                continue
            if t.catalog or (t.db and t.db.lower() != 'dbo') or nombre not in TABLAS_PERMITIDAS:
                return False, (f'Tabla no permitida: "{t.sql()}". Tablas disponibles: '
                               f'{", ".join(sorted(TABLAS_PERMITIDAS))}.')
    return True, ''


def _nombres_unicos(columnas):
    """SQL Server permite columnas sin nombre o repetidas; la tabla de pantalla necesita nombres únicos y no vacíos."""
    vistos, salida = {}, []
    for i, c in enumerate(columnas, 1):
        base = c if c else f'columna_{i}'
        n = vistos.get(base.lower(), 0) + 1
        vistos[base.lower()] = n
        salida.append(base if n == 1 else f'{base}_{n}')
    return salida


def _normalizar_resultado(r):
    for c in r.columns:
        if r[c].dtype == object and len(r):
            primero = r[c].dropna().iloc[0] if r[c].notna().any() else None
            if isinstance(primero, decimal.Decimal):
                r[c] = _numero(r[c])
            elif isinstance(primero, bool):
                r[c] = r[c].map(lambda v: None if v is None else int(v)).astype('Int64')
            elif isinstance(primero, (datetime.date, datetime.datetime, datetime.time)):
                r[c] = r[c].astype(str)
            elif isinstance(primero, (bytes, bytearray)):
                r[c] = r[c].map(lambda v: v.hex() if isinstance(v, (bytes, bytearray)) else v)
            elif isinstance(primero, uuid.UUID):
                r[c] = r[c].astype(str)
        elif pd.api.types.is_bool_dtype(r[c]):
            r[c] = r[c].astype(int)
        elif pd.api.types.is_datetime64_any_dtype(r[c]):
            r[c] = r[c].dt.strftime('%Y-%m-%d %H:%M:%S')
    return r


def ejecutar_sql(sql):
    """Ejecuta la consulta en SQL Server y devuelve (DataFrame, truncado). Lanza excepción si falla."""
    import pyodbc
    sql = sin_comentarios(sql).strip().rstrip(';').strip()          # se ejecuta exactamente lo que se validó
    conn = conectar(solo_lectura=True)
    conn.timeout = SQL_TIMEOUT                                    # timeout de la consulta
    try:
        cur = conn.cursor()
        cur.execute(sql)
        if cur.description is None:
            raise ValueError('La consulta no devolvió columnas.')
        columnas = _nombres_unicos([c[0] for c in cur.description])
        filas = cur.fetchmany(MAX_FILAS + 1)
    except pyodbc.Error as e:
        if str(e.args[0] if e.args else '') == 'HYT00' or 'timeout' in str(e).lower():
            raise ValueError(f'La consulta tardó más de {SQL_TIMEOUT} s y se canceló.') from e
        raise ValueError(limpiar_error_sql(e)) from e
    finally:
        conn.close()
    r = pd.DataFrame.from_records([tuple(f) for f in filas], columns=columnas)
    truncado = len(r) > MAX_FILAS
    return _normalizar_resultado(r.head(MAX_FILAS)), truncado


RESULTADOS = OrderedDict()                                        # caché en el servidor: el navegador solo guarda un identificador
_RESULTADOS_LOCK = threading.Lock()


def guardar_resultado(r):
    token = uuid.uuid4().hex
    with _RESULTADOS_LOCK:
        RESULTADOS[token] = r
        while len(RESULTADOS) > 20:                               # se conservan los 20 últimos
            RESULTADOS.popitem(last=False)
    return token


def columnas_tabla(r):
    cols = []
    for c in r.columns:
        if pd.api.types.is_float_dtype(r[c]):
            cols.append({'name': c, 'id': c, 'type': 'numeric', 'format': Format(precision=2, scheme=Scheme.fixed, group=Group.yes)})
        elif pd.api.types.is_numeric_dtype(r[c]) and not pd.api.types.is_bool_dtype(r[c]):
            cols.append({'name': c, 'id': c, 'type': 'numeric'})
        else:
            cols.append({'name': c, 'id': c})
    return cols


# =====================================================================
# 4. LAYOUT (DISEÑO DE LA PÁGINA)
# =====================================================================
# El layout es un ÁRBOL de componentes (html.* estructura, dcc.* controles y gráficos). Cada componente que participa
# en un callback lleva un "id" único. Estilos reutilizables, definidos una sola vez:
ESTILO_TARJETA = {'backgroundColor': 'white', 'padding': '14px', 'borderRadius': '8px', 'border': f'1px solid {BORDE}',
                  'boxShadow': '0 1px 3px rgba(1,3,177,0.07)', 'minWidth': 0}
ESTILO_GRAFICO = {'height': '420px'}                         # ALTURA FIJA: ver la nota de la sección 3c
ESTILO_GRILLA = {'display': 'grid', 'gridTemplateColumns': '1fr 1fr', 'gap': '15px', 'marginTop': '15px'}
ETIQUETA = {'fontWeight': 'bold', 'fontSize': '13px', 'color': TINTA_SEC}
ESTILO_TITULO_TARJETA = {'marginTop': 0, 'color': AZUL}
ESTILO_AYUDA = {'color': TINTA_SEC, 'fontSize': '13px'}
ESTILO_ENCABEZADO_TABLA = {'backgroundColor': AZUL, 'color': 'white', 'fontWeight': 'bold', 'fontFamily': 'Arial, sans-serif'}
ESTILO_BOTON = {'padding': '8px 16px', 'border': 'none', 'borderRadius': '6px', 'cursor': 'pointer', 'fontWeight': 'bold'}
ESTILO_BOTON_PRIMARIO = {**ESTILO_BOTON, 'backgroundColor': AZUL, 'color': 'white'}
ESTILO_BOTON_SECUNDARIO = {**ESTILO_BOTON, 'backgroundColor': 'white', 'color': AZUL, 'border': f'1.5px solid {AZUL}'}


def control(titulo, componente):
    return html.Div([html.Label(titulo, style=ETIQUETA), componente], style={'minWidth': 0})


def tarjeta_grafico(id_grafico, altura='420px', config=None):
    """Tarjeta con un gráfico dentro (siempre con altura fija)."""
    return html.Div(dcc.Graph(id=id_grafico, style={'height': altura}, config=config or {'displaylogo': False}), style=ESTILO_TARJETA)


CONFIG_MAPA = {'scrollZoom': True, 'displaylogo': False, 'modeBarButtonsToRemove': ['select2d', 'lasso2d', 'autoScale2d']}


def tarjeta_kpi(valor, titulo, detalle='', color=TINTA, color_detalle=TINTA_SEC, destacada=False):
    """Tarjeta KPI: número grande + título + línea de detalle."""
    return html.Div([
        html.Div(valor, style={'fontSize': '26px', 'fontWeight': 'bold', 'color': color}),
        html.Div(titulo, style={'color': TINTA_SEC, 'fontSize': '13px'}),
        html.Div(detalle, style={'color': color_detalle, 'fontSize': '12px', 'marginTop': '4px', 'minHeight': '16px',
                                 'fontWeight': 'bold' if color_detalle != TINTA_SEC else 'normal'}),
    ], style={**ESTILO_TARJETA, 'textAlign': 'center', 'borderTop': f'3px solid {AZUL if destacada else BORDE}'})


ENTERO = Format(precision=0, scheme=Scheme.fixed, group=Group.yes)
UN_DECIMAL = Format(precision=1, scheme=Scheme.fixed)
CON_SIGNO = Format(precision=1, scheme=Scheme.fixed, sign=Sign.positive)

COLUMNAS_DISTRITOS = [
    {'name': 'Distrito', 'id': 'distrito'}, {'name': 'Provincia', 'id': 'provincia'}, {'name': 'Oficina', 'id': 'oficina'},
    {'name': 'Bolsas (actual)', 'id': 'vol_ult', 'type': 'numeric', 'format': ENTERO},
    {'name': 'Share anterior %', 'id': 's_ref', 'type': 'numeric', 'format': UN_DECIMAL},
    {'name': 'Share actual %', 'id': 's_ult', 'type': 'numeric', 'format': UN_DECIMAL},
    {'name': 'Cambio (pp)', 'id': 'delta', 'type': 'numeric', 'format': CON_SIGNO},
    {'name': 'Bolsas por recuperar', 'id': 'recuperable', 'type': 'numeric', 'format': ENTERO}]
COLUMNAS_PRIORIDAD = [
    {'name': 'Prioridad', 'id': 'prioridad'}, {'name': 'Oficina', 'id': 'oficina'},
    {'name': 'Share anterior %', 'id': 's_ref', 'type': 'numeric', 'format': UN_DECIMAL},
    {'name': 'Share actual %', 'id': 's_ult', 'type': 'numeric', 'format': UN_DECIMAL},
    {'name': 'Cambio (pp)', 'id': 'delta', 'type': 'numeric', 'format': CON_SIGNO},
    {'name': 'Bolsas (actual)', 'id': 'vol_ult', 'type': 'numeric', 'format': ENTERO},
    {'name': 'Bolsas por recuperar', 'id': 'recuperable', 'type': 'numeric', 'format': ENTERO},
    {'name': 'Competidor que más ganó', 'id': 'competidor'}, {'name': 'Motivo principal', 'id': 'motivo'}]

ESTILO_CELDA = {'textAlign': 'left', 'padding': '6px', 'fontSize': '13px', 'fontFamily': 'Arial, sans-serif'}
ESTILO_CELDA_TEXTO = {**ESTILO_CELDA, 'whiteSpace': 'normal', 'height': 'auto', 'lineHeight': '1.35'}
FILA_IMPAR = {'if': {'row_index': 'odd'}, 'backgroundColor': '#f7f8fd'}
COLOR_ESTADO = [{'if': {'filter_query': f'{{estado}} = "{e}"', 'column_id': 'estado'}, 'color': c, 'fontWeight': 'bold'}
                for e, c in [('En meta', VERDE_TEXTO), ('Alerta', AMBAR_TEXTO), ('Crítico', ROJO_TEXTO)]]


def panel_esquema():
    """Panel lateral de la pestaña SQL: qué tablas y columnas se pueden consultar."""
    def fila(c, t, d):
        return html.Div([html.Code(c, style={'fontWeight': 'bold'}), html.Span(f'  {t}', style={'color': TINTA_SUAVE, 'fontSize': '11px'}),
                         html.Div(d, style={'color': TINTA_SEC, 'fontSize': '11px'}) if d else None],
                        style={'padding': '2px 0', 'borderBottom': '1px solid #eceef7'})
    bloques = [html.Details(open=True, children=[
        html.Summary('vw_consumo  (vista plana, recomendada)', style={'fontWeight': 'bold', 'cursor': 'pointer', 'color': AZUL}),
        html.Div([fila(*x) for x in DICCIONARIO_VISTA], style={'maxHeight': '360px', 'overflowY': 'auto', 'marginTop': '6px'})])]
    for tabla, columnas in ESQUEMA_ESTRELLA.items():
        bloques.append(html.Details([html.Summary(tabla, style={'cursor': 'pointer'}),
                                     html.P(columnas, style={'fontSize': '11px', 'color': TINTA_SEC, 'margin': '4px 0'})]))
    return html.Div([html.H4('Tablas y columnas', style=ESTILO_TITULO_TARJETA), *bloques], style=ESTILO_TARJETA)


# ---------------------------------------------------------------------
# BARRA DE FILTROS (compartida por las pestañas 1 a 4)
# ---------------------------------------------------------------------
barra_filtros = html.Div(id='barra-filtros', style={**ESTILO_TARJETA, 'display': 'grid', 'gap': '14px',
                                                    'gridTemplateColumns': 'repeat(4, minmax(0, 1fr))', 'marginBottom': '15px'}, children=[
    control('Periodo analizado (últimos meses)',
            dcc.RadioItems(id='f-ventana', options=[{'label': f'{n} meses', 'value': n} for n in (3, 6, 9, 12)], value=VENTANA_DEF, inline=True,
                           inputStyle={'marginLeft': '10px', 'marginRight': '4px'})),
    control('Comparar contra',
            dcc.RadioItems(id='f-comparar', options=[{'label': 'Periodo anterior', 'value': 'prev'},
                                                     {'label': 'Mismo periodo año anterior', 'value': 'yoy'}], value='prev', inline=True,
                           inputStyle={'marginLeft': '10px', 'marginRight': '4px'})),
    control('Meta de share (%) — vacío = nivel del año anterior',
            dcc.Input(id='f-meta', type='number', min=0, max=100, step=0.5, debounce=True, placeholder='automática',
                      style={'width': '100%', 'padding': '7px', 'boxSizing': 'border-box', 'border': f'1px solid {BORDE}', 'borderRadius': '4px'})),
    control('Volumen mínimo por distrito (mapa y tabla)',
            dcc.Slider(id='f-minvol', min=0, max=1500, step=50, value=MIN_VOLUMEN_DEF, marks={0: '0', 500: '500', 1000: '1k', 1500: '1.5k'},
                       tooltip={'placement': 'bottom', 'always_visible': False})),
    control('Departamento', dcc.Dropdown(id='f-depto', options=OPC_DEPTO, multi=True, placeholder='Todos')),
    control('Oficina', dcc.Dropdown(id='f-oficina', options=OPC_OFICINA, multi=True, placeholder='Todas')),
    control('Etapa de obra', dcc.Dropdown(id='f-etapa', options=OPC_ETAPA, multi=True, placeholder='Todas')),
    control('Tipo de ejecución de la obra', dcc.Dropdown(id='f-ejec', options=OPC_EJEC, multi=True, placeholder='Todos')),
])
ENTRADAS_FILTROS = [Input('f-ventana', 'value'), Input('f-comparar', 'value'), Input('f-depto', 'value'),
                    Input('f-oficina', 'value'), Input('f-etapa', 'value'), Input('f-ejec', 'value')]

# ---------------------------------------------------------------------
# PESTAÑA 1: RESUMEN Y KPIs
# ---------------------------------------------------------------------
tab_resumen = html.Div([
    html.Div(id='kpis', style={'display': 'grid', 'gridTemplateColumns': 'repeat(5, minmax(0, 1fr))', 'gap': '15px'}),
    html.Div(style={**ESTILO_TARJETA, 'marginTop': '15px', 'borderLeft': f'4px solid {ROJO}'}, children=[
        html.H4('Insights: lo más relevante con los filtros actuales', style=ESTILO_TITULO_TARJETA),
        html.Div(id='hallazgos')]),
    html.Div(style=ESTILO_GRILLA, children=[
        tarjeta_grafico('g-evol'), tarjeta_grafico('g-volumen'), tarjeta_grafico('g-puente'), tarjeta_grafico('g-anual')]),
])

# ---------------------------------------------------------------------
# PESTAÑA 2: ¿DÓNDE Y CUÁNDO CAE?
# ---------------------------------------------------------------------
tab_donde = html.Div([
    html.Div(style={**ESTILO_TARJETA, 'display': 'grid', 'gridTemplateColumns': 'minmax(0, 1fr) minmax(0, 2fr)', 'gap': '15px', 'alignItems': 'center'}, children=[
        control('Ver el cambio del share por…',
                dcc.Dropdown(id='f-dimension', options=list(DIMENSIONES), value='Oficina', clearable=False)),
        html.P('El datamart no tiene un campo de canal de venta: el tipo de ejecución, la etapa de obra y la sociedad son los '
               'segmentos disponibles para aproximarlo. "Efecto tasa" = el share cayó dentro del segmento; '
               '"efecto mezcla" = cambió cuánto pesa el segmento en el censo.', style=ESTILO_AYUDA)]),
    html.Div(style=ESTILO_GRILLA, children=[
        tarjeta_grafico('g-delta', '460px'), tarjeta_grafico('g-aporte', '460px'),
        tarjeta_grafico('g-mapa', '640px', CONFIG_MAPA), tarjeta_grafico('g-heat', '640px')]),
    html.Div(style={**ESTILO_TARJETA, 'marginTop': '15px'}, children=[
        html.H4('Distritos con más bolsas por recuperar', style=ESTILO_TITULO_TARJETA),
        html.P('Solo distritos con volumen mínimo en ambos periodos. "Bolsas por recuperar" = bolsas que habría que volver a ganar '
               'para recuperar el share del periodo anterior.', style=ESTILO_AYUDA),
        dash_table.DataTable(id='t-distritos', columns=COLUMNAS_DISTRITOS, page_size=15, sort_action='native',
                             style_header=ESTILO_ENCABEZADO_TABLA, style_cell=ESTILO_CELDA,
                             style_data_conditional=[FILA_IMPAR,
                                                     {'if': {'filter_query': '{delta} < -10', 'column_id': 'delta'}, 'color': ROJO_TEXTO, 'fontWeight': 'bold'}])]),
])

# ---------------------------------------------------------------------
# PESTAÑA 3: COMPETENCIA Y PRECIO
# ---------------------------------------------------------------------
tab_competencia = html.Div([
    html.Div(style={**ESTILO_GRILLA, 'marginTop': 0}, children=[
        tarjeta_grafico('g-marcas'), tarjeta_grafico('g-competidoras'),
        tarjeta_grafico('g-precio'), tarjeta_grafico('g-precio-caida', '440px'),
        tarjeta_grafico('g-motivos', '440px')]),
])

# ---------------------------------------------------------------------
# PESTAÑA 4: PROPUESTA
# ---------------------------------------------------------------------
tab_propuesta = html.Div([
    html.Div(id='objetivo', style={**ESTILO_TARJETA, 'borderLeft': f'4px solid {ROJO}'}),
    html.Div(style=ESTILO_GRILLA, children=[
        html.Div(style=ESTILO_TARJETA, children=[
            html.H4('Simulador de recuperación', style=ESTILO_TITULO_TARJETA),
            html.P('Elija cuántas oficinas priorizar y qué parte de lo perdido se recupera en cada una.', style=ESTILO_AYUDA),
            control('Oficinas priorizadas (por bolsas por recuperar)',
                    dcc.Slider(id='sim-n', min=1, max=max(N_OFICINAS, 2), step=1, value=min(5, N_OFICINAS),
                               marks={i: str(i) for i in range(1, max(N_OFICINAS, 2) + 1, 2)}, tooltip={'placement': 'bottom'})),
            html.Div(style={'height': '10px'}),
            control('% de la caída que se recupera en cada oficina',
                    dcc.Slider(id='sim-pct', min=10, max=100, step=10, value=50, marks={i: f'{i}%' for i in range(10, 101, 30)},
                               tooltip={'placement': 'bottom'})),
            html.Div(id='sim-texto', style={'marginTop': '12px', 'fontWeight': 'bold', 'color': AZUL})]),
        tarjeta_grafico('g-sim', '420px')]),
    html.Div(style={**ESTILO_TARJETA, 'marginTop': '15px'}, children=[
        html.H4('Prioridad por oficina', style=ESTILO_TITULO_TARJETA),
        dash_table.DataTable(id='t-prioridad', columns=COLUMNAS_PRIORIDAD, page_size=15, sort_action='native',
                             style_header=ESTILO_ENCABEZADO_TABLA, style_cell=ESTILO_CELDA_TEXTO,
                             style_data_conditional=[FILA_IMPAR, {'if': {'filter_query': '{delta} < -3', 'column_id': 'delta'}, 'color': ROJO_TEXTO, 'fontWeight': 'bold'}])]),
    html.Div(style={**ESTILO_TARJETA, 'marginTop': '15px'}, children=[
        html.H4('Propuesta de acciones (hipótesis a validar con el equipo comercial)', style=ESTILO_TITULO_TARJETA),
        dash_table.DataTable(id='t-acciones', columns=[{'name': n, 'id': i} for n, i in [
            ('#', 'n'), ('Acción propuesta', 'accion'), ('Dónde', 'donde'), ('Evidencia en los datos', 'evidencia'), ('KPI de seguimiento y meta', 'kpi')]],
            style_header=ESTILO_ENCABEZADO_TABLA, style_cell=ESTILO_CELDA_TEXTO,
            style_cell_conditional=[{'if': {'column_id': 'n'}, 'width': '30px'}, {'if': {'column_id': 'accion'}, 'width': '30%'},
                                    {'if': {'column_id': 'evidencia'}, 'width': '30%'}], style_data_conditional=[FILA_IMPAR])]),
    html.Div(style={**ESTILO_TARJETA, 'marginTop': '15px'}, children=[
        html.H4('Indicadores de seguimiento oportuno', style=ESTILO_TITULO_TARJETA),
        html.P('Tablero propuesto para monitorear el share cada mes. Los umbrales se pueden ajustar en la configuración (UMBRAL_ALERTA, UMBRAL_CRITICO).', style=ESTILO_AYUDA),
        dash_table.DataTable(id='t-indicadores', columns=[{'name': n, 'id': i} for n, i in [
            ('Indicador', 'indicador'), ('Definición', 'definicion'), ('Frecuencia', 'frecuencia'), ('Valor actual', 'valor'),
            ('Umbral / meta', 'umbral'), ('Estado', 'estado')]],
            style_header=ESTILO_ENCABEZADO_TABLA, style_cell=ESTILO_CELDA_TEXTO, style_data_conditional=[FILA_IMPAR] + COLOR_ESTADO)]),
    html.Div(style={**ESTILO_TARJETA, 'marginTop': '15px'}, children=[
        html.H4('Alcance y limitaciones de los datos', style=ESTILO_TITULO_TARJETA), html.Div(id='alcance')]),
])


# ---------------------------------------------------------------------
# PESTAÑA 5: CONSULTAS SQL
# ---------------------------------------------------------------------
def layout_sql():
    primero = list(EJEMPLOS)[0]
    return html.Div(style={'display': 'grid', 'gridTemplateColumns': 'minmax(0, 3fr) minmax(0, 1fr)', 'gap': '15px'}, children=[
        html.Div([
            html.Div(style=ESTILO_TARJETA, children=[
                html.Div(style={'display': 'grid', 'gridTemplateColumns': 'minmax(0, 1fr) auto', 'gap': '10px', 'alignItems': 'end'}, children=[
                    control('Cargar una consulta de ejemplo',
                            dcc.Dropdown(id='sql-ejemplo', options=list(EJEMPLOS), placeholder='Elija un ejemplo...')),
                    html.Div(f'Solo lectura · máx. {MAX_FILAS:,} filas · {SQL_TIMEOUT} s',
                             style={'color': TINTA_SUAVE, 'fontSize': '12px', 'paddingBottom': '8px'}),
                ]),
                dcc.Textarea(id='sql-editor', value=EJEMPLOS[primero], spellCheck=False,
                             style={'width': '100%', 'height': '240px', 'marginTop': '10px', 'fontFamily': 'Consolas, monospace',
                                    'fontSize': '13px', 'padding': '10px', 'boxSizing': 'border-box', 'border': f'1px solid {BORDE}',
                                    'borderRadius': '6px'}),
                html.Div(style={'display': 'flex', 'gap': '10px', 'marginTop': '10px', 'alignItems': 'center', 'flexWrap': 'wrap'}, children=[
                    html.Button('▶ Ejecutar', id='sql-run', n_clicks=0, className='btn-primario', style=ESTILO_BOTON_PRIMARIO),
                    html.Button('Descargar CSV', id='sql-csv', n_clicks=0, className='btn-secundario', style=ESTILO_BOTON_SECUNDARIO),
                    html.Button('Descargar Excel', id='sql-xlsx', n_clicks=0, className='btn-secundario', style=ESTILO_BOTON_SECUNDARIO),
                    dcc.Download(id='sql-descarga'),
                    dcc.Store(id='sql-token'),
                ]),
            ]),
            html.Div(style={**ESTILO_TARJETA, 'marginTop': '15px'}, children=[
                dcc.Loading(type='circle', color=AZUL, children=[
                    html.Div(id='sql-msg', children='Escriba una consulta y presione Ejecutar.',
                             style={'marginBottom': '8px', 'fontSize': '13px', 'color': TINTA_SEC}),
                    dash_table.DataTable(
                        id='sql-tabla', page_size=20, sort_action='native', filter_action='native',
                        style_table={'overflowX': 'auto'}, style_header=ESTILO_ENCABEZADO_TABLA,
                        style_data_conditional=[FILA_IMPAR],
                        style_cell={**ESTILO_CELDA, 'maxWidth': '320px', 'overflow': 'hidden', 'textOverflow': 'ellipsis'}),
                ]),
            ]),
            html.Div(style={**ESTILO_TARJETA, 'marginTop': '15px'}, children=[
                html.H4('Gráfico rápido del resultado', style=ESTILO_TITULO_TARJETA),
                html.Div(style={'display': 'grid', 'gridTemplateColumns': 'repeat(3, minmax(0, 1fr))', 'gap': '12px'}, children=[
                    control('Eje X (categoría o fecha)', dcc.Dropdown(id='sql-x', clearable=False)),
                    control('Eje Y (número)', dcc.Dropdown(id='sql-y', clearable=False)),
                    control('Tipo', dcc.RadioItems(id='sql-tipo', options=['Barras', 'Líneas'], value='Barras', inline=True,
                                                   inputStyle={'marginLeft': '10px', 'marginRight': '4px'})),
                ]),
                dcc.Graph(id='sql-grafico', style={'height': '380px'}),
            ]),
        ], style={'minWidth': 0}),
        html.Div(panel_esquema(), style={'minWidth': 0}),
    ])


# ---------------------------------------------------------------------
# CREAR LA APP Y ARMAR EL LAYOUT GENERAL
# ---------------------------------------------------------------------
app = dash.Dash(__name__, assets_folder=RUTA_ASSETS)
app.title = TITULO

if USUARIO_WEB and CLAVE_WEB:
    import hmac
    from flask import request, Response

    @app.server.before_request
    def pedir_contrasena():
        """Autenticación básica HTTP: protege la página, los datos de los gráficos y la consola SQL."""
        a = request.authorization
        if a and hmac.compare_digest(a.username or '', USUARIO_WEB) and hmac.compare_digest(a.password or '', CLAVE_WEB):
            return None
        return Response('Acceso restringido', 401, {'WWW-Authenticate': 'Basic realm="Dashboard market share Cemex"'})
    print('🔒 Contraseña activada (DASH_USER / DASH_PASS).')
else:
    print('⚠ Sin contraseña: defina DASH_USER y DASH_PASS antes de compartir el dashboard por internet.')

# Ajustes de color de los componentes de Dash que no aceptan estilos en línea (selección, deslizador, hover)
app.index_string = '''<!DOCTYPE html>
<html>
    <head>
        {%metas%}
        <title>{%title%}</title>
        {%favicon%}
        {%css%}
        <style>
            :root { --cemex-azul: ''' + AZUL + '''; --cemex-rojo: ''' + ROJO + '''; --Dash-Fill-Interactive-Strong: ''' + AZUL + '''; }  /* Dash 4: color de selección y foco */
            body { margin: 0; background: ''' + FONDO + '''; }
            input[type="radio"], input[type="checkbox"] { accent-color: var(--cemex-azul); }
            .btn-primario:hover { background-color: #00008a !important; }
            .btn-secundario:hover { background-color: #eceefb !important; }
            button:focus-visible, a:focus-visible { outline: 3px solid var(--cemex-rojo); outline-offset: 2px; }
            .rc-slider-track { background-color: var(--cemex-azul) !important; }
            .rc-slider-handle { border-color: var(--cemex-azul) !important; background-color: var(--cemex-azul) !important; }
            .rc-slider-handle:active, .rc-slider-handle:focus { box-shadow: 0 0 0 5px rgba(1,3,177,0.2) !important; }
            .Select-value, .Select--multi .Select-value { background-color: #e6e7f8 !important; border-color: #c3c5ee !important; color: var(--cemex-azul) !important; }
            .Select-value-icon { border-color: #c3c5ee !important; }
            .Select-value-icon:hover { background-color: var(--cemex-rojo) !important; color: white !important; }
            .is-focused:not(.is-open) > .Select-control { border-color: var(--cemex-azul) !important; box-shadow: 0 0 0 1px var(--cemex-azul) !important; }
            .Select-option.is-focused { background-color: #eceefb !important; }
            .Select-option.is-selected { background-color: #dfe0f7 !important; color: var(--cemex-azul) !important; }
        </style>
    </head>
    <body>
        {%app_entry%}
        <footer>
            {%config%}
            {%scripts%}
            {%renderer%}
        </footer>
    </body>
</html>'''

ESTILO_PESTANA = {'padding': '10px 14px', 'fontWeight': 'bold', 'color': TINTA_SEC, 'backgroundColor': '#e9eaf6',
                  'border': f'1px solid {BORDE}', 'borderBottom': 'none', 'borderTopLeftRadius': '8px', 'borderTopRightRadius': '8px'}
ESTILO_PESTANA_ACTIVA = {**ESTILO_PESTANA, 'color': AZUL, 'backgroundColor': 'white', 'borderTop': f'3px solid {ROJO}'}


def encabezado():
    """Franja blanca con el logo de Cemex a la izquierda, el título y una línea roja de acento."""
    if os.path.exists(os.path.join(RUTA_ASSETS, ARCHIVO_LOGO)):
        logo = html.Img(src=app.get_asset_url(ARCHIVO_LOGO), alt='Cemex', style={'height': '46px', 'display': 'block'})
    else:
        logo = html.Div('CEMEX', style={'fontSize': '30px', 'fontWeight': '900', 'color': AZUL, 'letterSpacing': '2px'})
    return html.Div(style={'backgroundColor': 'white', 'borderBottom': f'4px solid {ROJO}', 'boxShadow': '0 1px 4px rgba(1,3,177,0.12)'}, children=[
        html.Div(style={'maxWidth': '1500px', 'margin': '0 auto', 'padding': '12px 20px', 'display': 'flex',
                        'alignItems': 'center', 'gap': '22px', 'flexWrap': 'wrap'}, children=[
            logo,
            html.Div(style={'borderLeft': f'2px solid {BORDE}', 'paddingLeft': '22px'}, children=[
                html.H1(TITULO, style={'color': AZUL, 'margin': 0, 'fontSize': '26px'}),
                html.Div(f'¿Por qué cae el share? · {len(df):,} registros de censo de {PRODUCTO.lower()} ({UNIDAD}) · '
                         f'{FECHA_MIN:%d/%m/%Y} a {FECHA_MAX:%d/%m/%Y} · fuente: SQL Server ({BASE_DATOS})',
                         style={'color': TINTA_SUAVE, 'fontSize': '13px', 'marginTop': '2px'}),
            ]),
        ])])


app.layout = html.Div(style={'fontFamily': 'Arial, sans-serif', 'backgroundColor': FONDO, 'minHeight': '100vh', 'color': TINTA}, children=[
    encabezado(),
    html.Div(style={'padding': '20px', 'maxWidth': '1500px', 'margin': '0 auto'}, children=[
        dcc.Tabs(id='tabs', value='tab-resumen', colors={'border': BORDE, 'primary': ROJO, 'background': '#e9eaf6'}, children=[
            dcc.Tab(label='1. Resumen y KPIs', value='tab-resumen', style=ESTILO_PESTANA, selected_style=ESTILO_PESTANA_ACTIVA,
                    children=[html.Div(tab_resumen, style={'marginTop': '15px'})]),
            dcc.Tab(label='2. ¿Dónde y cuándo cae?', value='tab-donde', style=ESTILO_PESTANA, selected_style=ESTILO_PESTANA_ACTIVA,
                    children=[html.Div(tab_donde, style={'marginTop': '15px'})]),
            dcc.Tab(label='3. Competencia y precio', value='tab-competencia', style=ESTILO_PESTANA, selected_style=ESTILO_PESTANA_ACTIVA,
                    children=[html.Div(tab_competencia, style={'marginTop': '15px'})]),
            dcc.Tab(label='4. Propuesta', value='tab-propuesta', style=ESTILO_PESTANA, selected_style=ESTILO_PESTANA_ACTIVA,
                    children=[html.Div(tab_propuesta, style={'marginTop': '15px'})]),
            dcc.Tab(label='5. Consultas SQL', value='tab-sql', style=ESTILO_PESTANA, selected_style=ESTILO_PESTANA_ACTIVA,
                    children=[html.Div(layout_sql(), style={'marginTop': '15px'})]),
        ]),
    ]),
])
# La barra de filtros va antes de las pestañas pero se oculta en la pestaña SQL (ver callback).
app.layout.children[1].children.insert(0, barra_filtros)


# =====================================================================
# 5. CALLBACKS
# =====================================================================
# Un CALLBACK es una función que Dash ejecuta AUTOMÁTICAMENTE cuando cambia un Input.
# Los parámetros llegan EN EL ORDEN de los Input; se devuelven tantos valores como Output, en el mismo orden.
# Los 4 callbacks de análisis comparten ctx_desde(): calcula los periodos una vez y los guarda en caché.

# --- Mostrar la barra de filtros solo en las pestañas de análisis ---
@app.callback(Output('barra-filtros', 'style'), Input('tabs', 'value'))
def mostrar_filtros(tab):
    return {**barra_filtros.style, 'display': 'none' if tab == 'tab-sql' else 'grid'}


# --- PESTAÑA 1: KPIs, insights y 4 gráficos ---
@app.callback(
    Output('kpis', 'children'), Output('hallazgos', 'children'),
    Output('g-evol', 'figure'), Output('g-volumen', 'figure'), Output('g-puente', 'figure'), Output('g-anual', 'figure'),
    *ENTRADAS_FILTROS, Input('f-meta', 'value'))
def actualizar_resumen(ventana, comparar, deptos, oficinas, etapas, ejecs, meta):
    ctx = ctx_desde(ventana, comparar, deptos, oficinas, etapas, ejecs)
    if ctx.ult.empty:
        v = fig_vacia()
        return [tarjeta_kpi('—', 'Sin datos')], html.P('No hay registros con estos filtros.'), v, v, v, v
    k = calcular_kpis(ctx, meta)
    sh = k.sh
    kpis = [
        tarjeta_kpi(f'{sh:.1f}%', f'Share de volumen · {ctx.ult_txt}', f'● {k.estado}', AZUL, k.color_estado, True),
        tarjeta_kpi('—' if not _ok(k.meta) else f'{k.meta:.1f}%', 'Meta de share',
                    '' if not _ok(k.brecha) else f'Brecha: {pp(k.brecha)}', TINTA, k.color_estado),
        tarjeta_kpi(pp(sh - k.sh_prev), 'vs periodo anterior', f'{k.sh_prev:.1f}% en {ctx.prev_txt}' if _ok(k.sh_prev) else 'sin datos previos',
                    ROJO_TEXTO if _ok(k.sh_prev) and sh - k.sh_prev < -0.05 else TINTA),
        tarjeta_kpi(pp(sh - k.sh_yoy), 'vs mismo periodo año anterior', f'{k.sh_yoy:.1f}% en {ctx.yoy_txt}' if _ok(k.sh_yoy) else 'sin datos del año anterior',
                    ROJO_TEXTO if _ok(k.sh_yoy) and sh - k.sh_yoy < -0.05 else TINTA),
        tarjeta_kpi(pp(k.ma3 - k.pico) if _ok(k.ma3) and _ok(k.pico) else '—', 'Caída desde el máximo (12 meses)',
                    f'Media móvil 3M: {k.pico:.1f}% ({etiqueta_periodo(k.pico_mes, k.pico_mes)}) → {k.ma3:.1f}%' if _ok(k.pico) and _ok(k.ma3) else '',
                    ROJO_TEXTO if _ok(k.pico) and _ok(k.ma3) and k.ma3 - k.pico < -0.5 else TINTA),
        tarjeta_kpi('—' if not _ok(k.g_propio) else f'{k.g_propio:+.1f}%', 'Crecimiento de bolsas de Cemex',
                    f'Mercado {k.g_mercado:+.1f}% · Competencia {k.g_comp:+.1f}%' if _ok(k.g_mercado) else 'sin referencia',
                    ROJO_TEXTO if _ok(k.g_propio) and _ok(k.g_mercado) and k.g_propio < k.g_mercado else TINTA),
        tarjeta_kpi('—' if not _ok(k.obras) else f'{k.obras:.1f}%', 'Penetración en obras',
                    f'{pp(k.obras - k.obras_ref)} vs referencia' if _ok(k.obras) and _ok(k.obras_ref) else '%  de obras donde aparece Cemex'),
        tarjeta_kpi('—' if not _ok(k.idx_precio) else f'{k.idx_precio:.0f}', 'Índice de precio (100 = competencia)',
                    f'S/ {k.p_propio:.2f} vs S/ {k.p_comp:.2f}' if _ok(k.idx_precio) else ''),
        tarjeta_kpi(f'{k.n_ofi_alerta} de {k.n_ofi_datos}', 'Oficinas en alerta', f'caída de {UMBRAL_OFICINA:.0f} pp o más',
                    ROJO_TEXTO if k.n_ofi_alerta > 0 else TINTA),
        tarjeta_kpi('—' if not _ok(k.rec_meta) else formato(k.rec_meta), f'{NOMBRE_VALOR.capitalize()} por recuperar',
                    'para llegar a la meta en este periodo'),
    ]
    hall = html.Ul([html.Li(x, style={'marginBottom': '5px'}) for x in generar_hallazgos(ctx, k)], style={'margin': 0, 'paddingLeft': '20px'})
    return kpis, hall, graf_evolucion_share(ctx, k), graf_volumen(ctx), graf_puente(k.t_ofi, ctx), graf_anual(ctx)


# --- PESTAÑA 2: ¿dónde y cuándo cae? ---
@app.callback(
    Output('g-delta', 'figure'), Output('g-aporte', 'figure'), Output('g-mapa', 'figure'), Output('g-heat', 'figure'),
    Output('t-distritos', 'data'),
    *ENTRADAS_FILTROS, Input('f-minvol', 'value'), Input('f-dimension', 'value'))
def actualizar_donde(ventana, comparar, deptos, oficinas, etapas, ejecs, minvol, dimension):
    ctx = ctx_desde(ventana, comparar, deptos, oficinas, etapas, ejecs)
    if ctx.ult.empty or ctx.ref is None:
        v = fig_vacia('Sin datos o sin periodo de comparación para estos filtros')
        return v, v, v, v, []
    col = DIMENSIONES.get(dimension, 'oficina')
    t = tabla_segmento(ctx.ult, ctx.ref, col)
    td = tabla_distritos(ctx)
    tabla = []
    if len(td):
        x = td[(td.vol_ult >= minvol) & (td.vol_ref >= minvol)].dropna(subset=['delta'])
        tabla = x.sort_values('recuperable', ascending=False).head(30)[
            ['distrito', 'provincia', 'oficina', 'vol_ult', 's_ref', 's_ult', 'delta', 'recuperable']].round(1).to_dict('records')
    return (graf_delta_segmento(t, dimension), graf_aporte_segmento(t, dimension),
            graf_mapa_delta(td, minvol, float(ctx.ult.volumen.sum())), graf_heatmap(ctx), tabla)


# --- PESTAÑA 3: competencia y precio ---
@app.callback(
    Output('g-marcas', 'figure'), Output('g-competidoras', 'figure'), Output('g-precio', 'figure'),
    Output('g-precio-caida', 'figure'), Output('g-motivos', 'figure'),
    *ENTRADAS_FILTROS)
def actualizar_competencia(ventana, comparar, deptos, oficinas, etapas, ejecs):
    ctx = ctx_desde(ventana, comparar, deptos, oficinas, etapas, ejecs)
    if ctx.ult.empty:
        v = fig_vacia()
        return v, v, v, v, v
    sin_ref = fig_vacia('Sin periodo de comparación para estos filtros')
    return (graf_cambio_marcas(cambio_marcas(ctx.ult, ctx.ref)) if ctx.ref is not None else sin_ref,
            graf_competidoras(ctx), graf_precio(ctx),
            graf_precio_vs_caida(ctx) if ctx.ref is not None else sin_ref,
            graf_motivos(ctx.ult, ctx.ref, ctx) if ctx.ref is not None else sin_ref)


# --- PESTAÑA 4: propuesta ---
@app.callback(
    Output('objetivo', 'children'), Output('g-sim', 'figure'), Output('sim-texto', 'children'),
    Output('t-prioridad', 'data'), Output('t-acciones', 'data'), Output('t-indicadores', 'data'), Output('alcance', 'children'),
    *ENTRADAS_FILTROS, Input('f-meta', 'value'), Input('sim-n', 'value'), Input('sim-pct', 'value'))
def actualizar_propuesta(ventana, comparar, deptos, oficinas, etapas, ejecs, meta, n, pct):
    ctx = ctx_desde(ventana, comparar, deptos, oficinas, etapas, ejecs)
    if ctx.ult.empty or ctx.ref is None:
        return html.P('Sin datos o sin periodo de comparación para estos filtros.'), fig_vacia(), '', [], [], [], ''
    k = calcular_kpis(ctx, meta)
    prio = tabla_prioridad(ctx)
    objetivo = [html.H4('Objetivo: recuperar el market share', style=ESTILO_TITULO_TARJETA)]
    if _ok(k.brecha) and k.brecha < 0:
        objetivo.append(html.P(
            f'Cemex tiene {k.sh:.1f}% de share en {ctx.ult_txt}; la meta es {k.meta:.1f}%. Cerrar la brecha de {-k.brecha:.1f} pp equivale a '
            f'recuperar unas {formato(k.rec_meta)} bolsas por cada {ctx.ventana} meses ({k.estado}).', style={'fontSize': '15px', 'margin': 0}))
    else:
        objetivo.append(html.P(f'Cemex está en la meta ({k.sh:.1f}% vs {k.meta:.1f}%). El objetivo es sostenerla y vigilar las oficinas en alerta.',
                               style={'fontSize': '15px', 'margin': 0}))
    n = int(n or 5)
    pct = int(pct or 50)
    gan = pct / 100 * prio.head(n).recuperable.sum() / k.vol_total * 100 if len(prio) and k.vol_total > 0 else 0.0
    bolsas = pct / 100 * prio.head(n).recuperable.sum() if len(prio) else 0.0
    texto = f'+{gan:.2f} pp de share (≈ {formato(bolsas)} bolsas)'
    if _ok(k.brecha) and k.brecha < 0 and gan > 0:
        texto += f' = {gan / -k.brecha * 100:.0f}% de la brecha a la meta'
    tabla_prio = []
    if len(prio):
        p = prio.reset_index().rename(columns={'index': 'oficina'})
        p.insert(0, 'prioridad', range(1, len(p) + 1))
        tabla_prio = p[['prioridad', 'oficina', 's_ref', 's_ult', 'delta', 'vol_ult', 'recuperable', 'competidor', 'motivo']].round(1).to_dict('records')
    acciones = [dict(zip(['n', 'accion', 'donde', 'evidencia', 'kpi'], a)) for a in generar_acciones(ctx, k, prio)]
    _, cob_u, _ = tabla_motivos(ctx.ult, ctx.ref)
    alcance = html.Ul([html.Li(x, style={'marginBottom': '4px'}) for x in [
        f'Fuente: censo de obras de autoconstrucción (no despachos facturados). El share es la participación de Cemex en el volumen censado, en bolsas.',
        f'Periodo disponible: {etiqueta_periodo(MES_MIN, MES_FIN)}. No hay datos de 2021-2022 ni de 2026, por lo que no se puede analizar ese rango.',
        'Canal de venta: el datamart no tiene un campo de canal (el proveedor viene vacío). Se aproxima con tipo de ejecución, etapa de obra y sociedad; '
        'se recomienda registrar el canal en el censo.',
        f'Motivo de compra: solo está informado en {cob_u:.0f}% del volumen de competencia; las conclusiones sobre motivos son indicativas.',
        'La cobertura del censo cambia entre periodos (efecto mezcla). Por eso se separa la caída real (efecto tasa) del cambio de peso de cada oficina.',
        'Las acciones propuestas son hipótesis a partir de los datos: validar con el equipo comercial antes de ejecutarlas.']],
        style={'margin': 0, 'paddingLeft': '20px', 'fontSize': '14px'})
    return (objetivo, graf_simulador(prio, pct, n, k), texto, tabla_prio, acciones, tabla_indicadores(ctx, k), alcance)


# =====================================================================
# 5b. CALLBACKS DE CONSULTAS SQL (pestaña 5)
# =====================================================================
@app.callback(Output('sql-editor', 'value'), Input('sql-ejemplo', 'value'), prevent_initial_call=True)
def cargar_ejemplo(nombre):
    return EJEMPLOS.get(nombre, dash.no_update)


@app.callback(
    Output('sql-msg', 'children'), Output('sql-tabla', 'data'), Output('sql-tabla', 'columns'),
    Output('sql-token', 'data'), Output('sql-x', 'options'), Output('sql-x', 'value'),
    Output('sql-y', 'options'), Output('sql-y', 'value'),
    Input('sql-run', 'n_clicks'), State('sql-editor', 'value'), prevent_initial_call=True)
def correr_consulta(clics, sql):
    def error(texto):
        return (html.Span('✖ ' + texto, style={'color': ROJO_TEXTO, 'fontWeight': 'bold'}), [], [], None, [], None, [], None)

    ok, motivo = validar_consulta(sql)
    if not ok:
        return error(motivo)
    t0 = time.time()
    try:
        r, truncado = ejecutar_sql(sql)
    except Exception as e:
        return error(f'La consulta falló: {str(e).splitlines()[0][:300] if str(e) else type(e).__name__}')
    seg = time.time() - t0
    token = guardar_resultado(r)
    msg = f'✔ {len(r):,} filas · {len(r.columns)} columnas · {seg:.2f} s'
    if truncado:
        msg += f'  ⚠ Resultado cortado a {MAX_FILAS:,} filas: agregue filtros o agrupe para ver todo.'
    if len(r) > MAX_FILAS_TABLA:
        msg += f'  Se muestran las primeras {MAX_FILAS_TABLA:,}; la descarga trae todas.'
    if len(r) == 0:
        msg = f'✔ La consulta no devolvió filas ({seg:.2f} s).'
    mostrado = r.head(MAX_FILAS_TABLA)
    datos = mostrado.astype(object).where(mostrado.notna(), None).to_dict('records')
    numericas = [c for c in r.columns if pd.api.types.is_numeric_dtype(r[c]) and not pd.api.types.is_bool_dtype(r[c])]
    x_def = next((c for c in r.columns if c not in numericas), r.columns[0] if len(r.columns) else None)
    y_def = next((c for c in reversed(numericas) if c != x_def), None)      # la última columna numérica suele ser la medida
    return (msg, datos, columnas_tabla(r), token, list(r.columns), x_def, numericas, y_def)


@app.callback(Output('sql-descarga', 'data'),
              Input('sql-csv', 'n_clicks'), Input('sql-xlsx', 'n_clicks'), State('sql-token', 'data'),
              prevent_initial_call=True)
def descargar_resultado(n_csv, n_xlsx, token):
    r = RESULTADOS.get(token)
    if r is None:
        return dash.no_update
    nombre = 'consulta_' + time.strftime('%Y%m%d_%H%M%S')
    if dash.ctx.triggered_id == 'sql-xlsx':
        return dcc.send_data_frame(r.to_excel, nombre + '.xlsx', index=False, sheet_name='Resultado')
    return dcc.send_data_frame(r.to_csv, nombre + '.csv', index=False, encoding='utf-8-sig')   # utf-8-sig: Excel abre bien las tildes


@app.callback(Output('sql-grafico', 'figure'),
              Input('sql-token', 'data'), Input('sql-x', 'value'), Input('sql-y', 'value'), Input('sql-tipo', 'value'))
def grafico_resultado(token, x, y, tipo):
    r = RESULTADOS.get(token)
    if r is None or x is None or y is None or x not in r.columns or y not in r.columns:
        return fig_vacia('Ejecute una consulta con una columna numérica para graficar')
    d = r.head(MAX_FILAS_TABLA)
    if tipo == 'Barras':
        fig = px.bar(d, x=x, y=y, color_discrete_sequence=[AZUL])
    else:
        fig = px.line(d, x=x, y=y, markers=True, color_discrete_sequence=[AZUL])
        fig.update_traces(line_width=2, marker_size=7, marker_line_color='white', marker_line_width=2)
    if not pd.api.types.is_numeric_dtype(d[x]):
        fig.update_xaxes(type='category')
    return estilo(fig, f'{y} por {x}', leyenda=False)


# =====================================================================
# 6. INICIAR EL SERVIDOR
# =====================================================================
if __name__ == '__main__':
    print(f'Abrir en el navegador: http://localhost:{PUERTO}')
    app.run(debug=DEBUG, port=PUERTO)


Conectando a SQL Server (localhost / DatamartAutoconstruccion)...
✓ 197,896 filas leídas desde SQL Server
⚠ Sin contraseña: defina DASH_USER y DASH_PASS antes de compartir el dashboard por internet.
Abrir en el navegador: http://localhost:8054
